<a href="https://colab.research.google.com/github/KErarslan/3D-VSK-kriging/blob/main/3DSurfaceKriging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Prerequisite: Core Classes (from the original pipeline's Cell 1)

`KalgoorlieData`, `EmpiricalVariogram`, `VariogramModel`, `VariogramSurface3D`,
`OrdinaryKriging`, `PSDAnalysis` — copied in here so this notebook is fully
self-contained and does not depend on any other file being run first.
If you already ran the original pipeline notebook's Cell 1 in this same
session, this cell is redundant but harmless (it will simply redefine the
same classes).

In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║         3D-VSK: 3D Variogram Surface Kriging Pipeline        ║
║         Kalgoorlie Au Dataset — Reproducible Code            ║
╠══════════════════════════════════════════════════════════════╣
║  Referans : Erarslan (2000) — kongre bildirisi (temel)       ║
║  Geliştirme: Smooth 3D yüzey + LMC + gerçek 3D anizotropi  ║
║  Hedef    : Mathematical Geosciences / Computers & Geosci.   ║
╚══════════════════════════════════════════════════════════════╝

Kullanım (Google Colab veya lokal):
    python 3dvsk_pipeline.py

Çıktılar:
    figures/  — tüm görseller (PNG)
    results/  — metrik tabloları (CSV)
    params/   — fitted variogram parametreleri (JSON)
"""

# ── Bağımlılıklar ─────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
from scipy.interpolate import RBFInterpolator, RegularGridInterpolator
from scipy.linalg import solve
import json, os, warnings
warnings.filterwarnings('ignore')

# ── Çıktı klasörleri ─────────────────────────────────────────
# ── Çıktı dizini: Colab için değiştirebilirsiniz ──────────────
OUTPUT_DIR = "figures"   # Colab: "/content/drive/MyDrive/3dvsk/"
RESULTS_DIR = "results"
PARAMS_DIR  = "params"

for d in [OUTPUT_DIR, RESULTS_DIR, PARAMS_DIR]:
    os.makedirs(d, exist_ok=True)

# ═════════════════════════════════════════════════════════════
# BÖLÜM 0 — VERİ
# ═════════════════════════════════════════════════════════════
class KalgoorlieData:
    """
    Kalgoorlie Altın Projesi — Northern Zone
    20 AC sondaj, koordinatlar /100 düzeltmeli
    Kaynak: Riversgold Limited (akademik kullanım için modifiye)
    """
    SURVEY = {
        "NZAC146":{"x":4120.40,"y":8110.20,"z":162.40},
        "NZAC147":{"x":4130.10,"y":8141.40,"z":162.20},
        "NZAC148":{"x":4140.80,"y":8128.80,"z":162.10},
        "NZAC149":{"x":4151.00,"y":8133.10,"z":162.00},
        "NZAC150":{"x":4102.10,"y":8109.80,"z":161.90},
        "NZAC151":{"x":4141.20,"y":8157.10,"z":161.70},
        "NZAC152":{"x":4130.90,"y":8136.10,"z":161.60},
        "NZAC153":{"x":4151.00,"y":8110.60,"z":161.50},
        "NZAC154":{"x":4101.50,"y":8145.80,"z":161.30},
        "NZAC155":{"x":4111.30,"y":8130.90,"z":161.10},
        "NZAC156":{"x":4122.10,"y":8120.40,"z":162.30},
        "NZAC157":{"x":4113.40,"y":8151.20,"z":162.20},
        "NZAC158":{"x":4142.80,"y":8133.30,"z":162.10},
        "NZAC159":{"x":4153.10,"y":8151.60,"z":161.90},
        "NZAC160":{"x":4153.60,"y":8152.10,"z":161.80},
        "NZAC161":{"x":4142.90,"y":8122.70,"z":161.60},
        "NZAC162":{"x":4123.40,"y":8130.30,"z":161.50},
        "NZAC163":{"x":4133.80,"y":8151.80,"z":161.30},
        "NZAC164":{"x":4103.20,"y":8140.10,"z":161.20},
        "NZAC165":{"x":4114.50,"y":8133.40,"z":161.00},
    }
    ASSAY = {
        "NZAC146":[2.010],"NZAC147":[1.250],"NZAC148":[1.100],
        "NZAC149":[0.610,0.880],"NZAC150":[2.090,1.750],
        "NZAC151":[0.630],"NZAC152":[1.190,1.220],
        "NZAC153":[1.650,1.110],"NZAC154":[1.720,1.410],
        "NZAC155":[1.470],"NZAC156":[1.880],
        "NZAC157":[1.470,1.620],"NZAC158":[0.880],
        "NZAC159":[1.110,1.260],"NZAC160":[1.010,1.330],
        "NZAC161":[1.390],"NZAC162":[1.610],
        "NZAC163":[0.450],"NZAC164":[1.630],
        "NZAC165":[1.910],
    }

    def __init__(self):
        self.hole_ids = sorted(self.SURVEY.keys())
        self.n        = len(self.hole_ids)
        self.coords   = np.array(
            [[self.SURVEY[h]["x"],self.SURVEY[h]["y"],self.SURVEY[h]["z"]]
              for h in self.hole_ids])
        self.au_raw   = np.array(
            [np.mean(self.ASSAY[h]) for h in self.hole_ids])
        self.au_log   = np.log(self.au_raw)

    def summary(self):
        print("─"*55)
        print("VERİ ÖZETİ — Kalgoorlie Au")
        print("─"*55)
        print(f"  n sondaj  : {self.n}")
        print(f"  Au g/t    : min={self.au_raw.min():.3f}, "
              f"max={self.au_raw.max():.3f}, "
              f"mean={self.au_raw.mean():.3f}")
        print(f"  ln(Au)    : mean={self.au_log.mean():.3f}, "
              f"std={self.au_log.std():.3f}")
        print(f"  X aralığı : {self.coords[:,0].min():.1f} – "
              f"{self.coords[:,0].max():.1f} m "
              f"(Δ={self.coords[:,0].max()-self.coords[:,0].min():.1f}m)")
        print(f"  Y aralığı : {self.coords[:,1].min():.1f} – "
              f"{self.coords[:,1].max():.1f} m "
              f"(Δ={self.coords[:,1].max()-self.coords[:,1].min():.1f}m)")
        print(f"  Z aralığı : {self.coords[:,2].min():.2f} – "
              f"{self.coords[:,2].max():.2f} m")


# ═════════════════════════════════════════════════════════════
# BÖLÜM 1 — VARİOGRAM MODELLERİ
# ═════════════════════════════════════════════════════════════
class VariogramModel:
    """Temel variogram/kovaryans fonksiyonları."""

    @staticmethod
    def spherical_gamma(h, C0, C, a):
        """Spherical variogram modeli γ(h)."""
        h = np.atleast_1d(np.array(h, dtype=float))
        g = np.where(h<=0, 0.0,
            np.where(h>=a, C0+C,
                     C0+C*(1.5*h/a - 0.5*(h/a)**3)))
        return g if g.size>1 else float(g.item())

    @staticmethod
    def spherical_cov(h, C0, C, a):
        """Spherical kovaryans fonksiyonu C(h) = sill - γ(h)."""
        sill = C0 + C
        if np.isscalar(h):
            if h <= 0:  return sill
            if h >= a:  return 0.0
            return sill - (C0 + C*(1.5*h/a - 0.5*(h/a)**3))
        h = np.array(h, dtype=float)
        return np.where(h<=0, sill,
               np.where(h>=a, 0.0,
                        sill-(C0+C*(1.5*h/a-0.5*(h/a)**3))))


# ═════════════════════════════════════════════════════════════
# BÖLÜM 2 — EMPİRİK VARİOGRAM VE MODEL FİTTİNG
# ═════════════════════════════════════════════════════════════
class EmpiricalVariogram:
    """
    Yönlü empirik variogram hesabı ve spherical model fitting.
    """
    DIRECTIONS = [0, 45, 90, 135]
    DIR_LABELS  = ["0° E-W","45° NE-SW","90° N-S","135° NW-SE"]

    def __init__(self, data: KalgoorlieData,
                 angle_tol=22.5, lag_width=8.0,
                 n_lags=7, lag_start=6.0):
        self.data      = data
        self.angle_tol = angle_tol
        self.lag_width = lag_width
        self.n_lags    = n_lags
        self.lag_start = lag_start
        self.empirical = {}   # {dir: {h, gamma, n_pairs}}
        self.fitted    = {}   # {dir: {C0, C, a, R2}}

    def _compute_direction(self, direction_deg):
        """Tek yön için empirik variogram."""
        coords = self.data.coords
        vals   = self.data.au_log
        n      = self.data.n
        dir_r  = np.radians(direction_deg)
        tol_r  = np.radians(self.angle_tol)
        lag_ctrs = self.lag_start + np.arange(self.n_lags)*self.lag_width

        h_out, g_out, np_out = [], [], []
        for lag_h in lag_ctrs:
            lo, hi = lag_h - self.lag_width/2, lag_h + self.lag_width/2
            sq = []
            for i in range(n):
                for j in range(i+1, n):
                    dx = coords[j,0]-coords[i,0]
                    dy = coords[j,1]-coords[i,1]
                    h  = np.sqrt(dx**2+dy**2)
                    if lo <= h < hi and h > 1e-6:
                        pair_ang = np.arctan2(dy, dx)
                        diffs = [abs(pair_ang-dir_r),
                                 abs(pair_ang-dir_r+np.pi),
                                 abs(pair_ang-dir_r-np.pi)]
                        if min(diffs) <= tol_r:
                            sq.append((vals[j]-vals[i])**2)
            if len(sq) >= 2:
                h_out.append(lag_h)
                g_out.append(np.mean(sq)/2.)
                np_out.append(len(sq))

        return (np.array(h_out), np.array(g_out), np.array(np_out))

    def compute_all(self):
        """Tüm yönler için empirik variogram hesapla."""
        for d in self.DIRECTIONS:
            h, g, np_ = self._compute_direction(d)
            self.empirical[d] = {"h":h, "gamma":g, "n_pairs":np_}
        return self

    def fit_all(self):
        """Her yön için spherical model fit et."""
        sill_init = np.var(self.data.au_log)
        for d in self.DIRECTIONS:
            res = self.empirical[d]
            h_d, g_d = res["h"], res["gamma"]
            if len(h_d) < 3:
                # Yetersiz veri — varsayılan
                self.fitted[d] = {"C0":0.01,"C":sill_init*0.9,
                                   "a":40.0,"R2":0.0}
                continue
            try:
                popt, _ = curve_fit(
                    VariogramModel.spherical_gamma, h_d, g_d,
                    p0=[sill_init*0.1, sill_init*0.9, h_d.max()*0.7],
                    bounds=([0,1e-6,1],[sill_init,sill_init*2,100]),
                    maxfev=5000)
                C0f,Cf,af = popt
                g_pred = VariogramModel.spherical_gamma(h_d,*popt)
                ss_res = np.sum((g_d-g_pred)**2)
                ss_tot = np.sum((g_d-g_d.mean())**2)
                r2 = 1-ss_res/ss_tot if ss_tot>0 else 0.0
                self.fitted[d] = {"C0":C0f,"C":Cf,"a":af,"R2":r2}
            except:
                self.fitted[d] = {"C0":0.01,"C":sill_init*0.9,
                                   "a":40.0,"R2":0.0}
        return self

    def print_summary(self):
        print("─"*60)
        print("EMPİRİK VARİOGRAM — Fitted Parametreler")
        print("─"*60)
        print(f"  {'Yön':<15} {'C0':>8} {'C':>8} {'a(m)':>10} "
              f"{'Sill':>8} {'R²':>8}")
        print("  "+"-"*53)
        for d,lbl in zip(self.DIRECTIONS,self.DIR_LABELS):
            p = self.fitted[d]
            print(f"  {lbl:<15} {p['C0']:>8.5f} {p['C']:>8.5f} "
                  f"{p['a']:>10.3f} {p['C0']+p['C']:>8.5f} {p['R2']:>8.4f}")

    def save_params(self, path=os.path.join(PARAMS_DIR,"variogram_params.json")):
        data = {str(d): self.fitted[d] for d in self.DIRECTIONS}
        with open(path,"w") as f:
            json.dump(data, f, indent=2)
        print(f"  Parametreler kaydedildi: {path}")


# ═════════════════════════════════════════════════════════════
# BÖLÜM 3 — 3D VARIOGRAM YÜZEYİ
# ═════════════════════════════════════════════════════════════
class VariogramSurface3D:
    """
    3D Variogram Surface — ana metodolojik katkı.

    Erarslan (2000): bilinear yüzey — yatay yönler
    3D-VSK         : LMC elipsoidal — azimuth + dip

    Kovaryans C(h, azimuth, dip) → sürekli fonksiyon
    """

    def __init__(self, ev: EmpiricalVariogram):
        self.ev   = ev
        self.dirs = ev.DIRECTIONS
        self._build_lmc_params()

    def _build_lmc_params(self):
        """LMC parametrelerini fitted değerlerden türet."""
        fp = self.ev.fitted
        # Sill normalizasyonu — PSD garantisi için
        sills = [fp[d]["C0"]+fp[d]["C"] for d in self.dirs]
        nugs  = [fp[d]["C0"] for d in self.dirs]
        self.sill_norm = np.mean(sills)
        self.nug_norm  = np.mean(nugs)
        self.b_nug     = max(0.010, self.nug_norm)
        self.b_str     = self.sill_norm - self.b_nug

        # Anizotropi parametreleri (range yöne göre)
        ranges = {d: fp[d]["a"] for d in self.dirs}
        self.a_min   = min(ranges.values())
        self.a_max   = max(ranges.values())
        self.theta_max = max(ranges, key=ranges.get)  # en uzun range yönü
        self.a_vert_f = 0.4   # dikey range faktörü

        print(f"\n  LMC Parametreleri:")
        print(f"    b_nugget  = {self.b_nug:.5f}")
        print(f"    b_struct  = {self.b_str:.5f}")
        print(f"    a_min     = {self.a_min:.3f}m  ({[d for d in self.dirs if ranges[d]==self.a_min][0]}°)")
        print(f"    a_max     = {self.a_max:.3f}m  ({self.theta_max}°)")
        print(f"    a_vert_f  = {self.a_vert_f}  (a_vert = a_horiz × {self.a_vert_f})")

    def a_ellipse(self, azimuth_deg):
        """Eliptik anizotropi: yöne bağlı range."""
        t  = np.radians(azimuth_deg % 180)
        tm = np.radians(self.theta_max)
        return self.a_min + (self.a_max-self.a_min)*np.cos(t-tm)**2

    # ── Kovaryans yöntemleri ───────────────────────────────

    def cov_bilinear(self, xi, xj):
        """
        Bilinear yüzey interpolasyonu — Erarslan (2000) yaklaşımı.
        Yalnızca yatay mesafe ve azimuth kullanır.
        """
        if np.allclose(xi,xj): return self.sill_norm
        fp   = self.ev.fitted
        h    = np.linalg.norm(xi[:2]-xj[:2])
        az   = np.degrees(np.arctan2(xj[1]-xi[1],xj[0]-xi[0])) % 180
        dirs = self.dirs + [180]
        vp_ext = {d: fp.get(d, fp[0]) for d in dirs}
        vp_ext[180] = fp[0]
        lo, hi = dirs[-2], dirs[-1]
        for k in range(len(dirs)-1):
            if dirs[k] <= az <= dirs[k+1]:
                lo, hi = dirs[k], dirs[k+1]; break
        p_lo = vp_ext[lo]; p_hi = vp_ext[hi]
        c_lo = VariogramModel.spherical_cov(h,p_lo["C0"],p_lo["C"],p_lo["a"])
        c_hi = VariogramModel.spherical_cov(h,p_hi["C0"],p_hi["C"],p_hi["a"])
        t = (az-lo)/(hi-lo) if hi!=lo else 0.0
        return (1-t)*c_lo + t*c_hi

    def cov_lmc_ellipsoidal(self, xi, xj):
        """
        3D-VSK ana yöntemi: LMC + elipsoidal anizotropi.
        Azimuth VE dip açısını birlikte ele alır.
        PSD yapısal olarak garantilidir (LMC).
        """
        if np.allclose(xi,xj): return self.b_nug + self.b_str
        dx = xj[0]-xi[0]; dy = xj[1]-xi[1]; dz = xj[2]-xi[2]
        h_h  = np.sqrt(dx**2+dy**2)
        az   = np.degrees(np.arctan2(dy,dx)) % 180
        a_h  = self.a_ellipse(az)
        a_v  = a_h * self.a_vert_f
        h_eff = np.sqrt((h_h/a_h)**2+(dz/a_v)**2)*a_h if a_h>1e-6 else abs(dz)
        # Spherical kovaryans (off-diagonal: nugget yok)
        if h_eff <= 0:    c = self.b_str
        elif h_eff >= a_h: c = 0.0
        else: c = self.b_str*(1-(1.5*h_eff/a_h-0.5*(h_eff/a_h)**3))
        return c

    def cov_diagonal(self):
        """Diagonal değer (i=i): nugget + sill."""
        return self.b_nug + self.b_str


# ═════════════════════════════════════════════════════════════
# BÖLÜM 4 — ORDINARY KRİGİNG
# ═════════════════════════════════════════════════════════════
class OrdinaryKriging:
    """
    Ordinary Kriging — 3D-VSK kovaryans fonksiyonu ile.
    """

    def __init__(self, surface: VariogramSurface3D):
        self.surf = surface

    def _build_system(self, coords_tr, cov_func):
        """Kriging sistem matrisi [C|1; 1^T|0]."""
        m  = len(coords_tr)
        diag = self.surf.cov_diagonal()
        K  = np.zeros((m+1,m+1))
        for i in range(m):
            K[i,i] = diag
            for j in range(i+1,m):
                v = cov_func(coords_tr[i],coords_tr[j])
                K[i,j] = v; K[j,i] = v
        K[:m,m] = 1.; K[m,:m] = 1.
        return K

    def predict(self, x0, coords_tr, vals_tr, cov_func):
        """
        Tek nokta tahmini.
        Döndürür: (z_hat, sigma2)
        """
        m  = len(coords_tr)
        K  = self._build_system(coords_tr, cov_func)
        k0 = np.array([cov_func(x0,coords_tr[i]) for i in range(m)]+[1.])
        try:
            w = solve(K, k0, assume_a='sym')
        except:
            w = np.linalg.lstsq(K, k0, rcond=None)[0]
        z_hat  = float(np.dot(w[:m], vals_tr))
        sigma2 = float(self.surf.cov_diagonal() - np.dot(w, k0))
        return z_hat, max(0., sigma2)

    def loocv(self, coords, vals, cov_func, label=""):
        """
        Leave-One-Out Cross Validation.
        Döndürür: dict ile tüm metrikler.
        """
        n = len(vals)
        preds = np.zeros(n)
        vars_ = np.zeros(n)
        for i in range(n):
            idx   = [j for j in range(n) if j!=i]
            z, s2 = self.predict(coords[i], coords[idx],
                                  vals[idx], cov_func)
            preds[i] = z; vars_[i] = s2

        res   = vals - preds
        ss_res = np.sum(res**2)
        ss_tot = np.sum((vals-vals.mean())**2)
        r2    = 1 - ss_res/ss_tot
        msdr  = float(np.mean(res**2/(vars_+1e-10)))
        # Back-transform
        z_bt  = np.exp(preds); v_bt = np.exp(vals)
        rmse_bt = np.sqrt(np.mean((v_bt-z_bt)**2))
        r2_bt   = 1-np.sum((v_bt-z_bt)**2)/np.sum((v_bt-v_bt.mean())**2)

        return {
            "label"  : label,
            "pred"   : preds,
            "var"    : vars_,
            "res"    : res,
            "ME"     : float(np.mean(res)),
            "MAE"    : float(np.mean(np.abs(res))),
            "RMSE"   : float(np.sqrt(np.mean(res**2))),
            "R2"     : float(r2),
            "MSDR"   : msdr,
            "RMSE_bt": rmse_bt,
            "R2_bt"  : r2_bt,
        }


# ═════════════════════════════════════════════════════════════
# BÖLÜM 5 — PSD ANALİZİ
# ═════════════════════════════════════════════════════════════
class PSDAnalysis:
    """Kovaryans matrisinin PSD kontrolü."""

    @staticmethod
    def build_matrix(coords, cov_offdiag, diag_val):
        n = len(coords)
        C = np.zeros((n,n))
        for i in range(n):
            C[i,i] = diag_val
            for j in range(i+1,n):
                v = cov_offdiag(coords[i],coords[j])
                C[i,j]=v; C[j,i]=v
        return C

    @staticmethod
    def check(C, label=""):
        eigs = np.linalg.eigvalsh(C)
        cond = np.linalg.cond(C)
        psd  = eigs.min() > -1e-10
        return {"label":label,"eigs":eigs,
                "lmin":eigs.min(),"lmax":eigs.max(),
                "cond":cond,"psd":psd}

    @staticmethod
    def print_result(res):
        sym = "✓" if res["psd"] else "✗"
        print(f"  {res['label']:<25} λ_min={res['lmin']:>9.5f}  "
              f"koşul={res['cond']:>8.2f}  PSD:{sym}")


# ═════════════════════════════════════════════════════════════
# BÖLÜM 6 — GÖRSELLEŞTİRME
# ═════════════════════════════════════════════════════════════
class Visualizer:
    """Pipeline görselleri."""

    @staticmethod
    def plot_data(data: KalgoorlieData):
        fig,axes=plt.subplots(1,2,figsize=(12,5))
        fig.suptitle("3D-VSK — Veri Özeti: Kalgoorlie Au",
                      fontsize=12,fontweight='bold')
        # Lokasyon haritası
        sc=axes[0].scatter(data.coords[:,0],data.coords[:,1],
                            c=data.au_log,cmap='YlOrRd',
                            s=100,edgecolors='k',linewidths=0.8,zorder=3)
        plt.colorbar(sc,ax=axes[0],label='ln(Au g/t)')
        for i,h in enumerate(data.hole_ids):
            axes[0].annotate(h[-4:],(data.coords[i,0],data.coords[i,1]),
                              fontsize=6,xytext=(0,4),
                              textcoords='offset points',ha='center')
        axes[0].set_xlabel('MGA East (m)'); axes[0].set_ylabel('MGA North (m)')
        axes[0].set_title('Sondaj Lokasyonları — ln(Au g/t)')
        axes[0].grid(True,alpha=0.3)
        # Histogram
        axes[1].hist(data.au_raw,bins=8,color='#e67e22',
                      edgecolor='k',alpha=0.8,label='Au g/t')
        ax2=axes[1].twinx()
        ax2.hist(data.au_log,bins=8,color='#3498db',
                  edgecolor='k',alpha=0.5,label='ln(Au)')
        axes[1].set_xlabel('Au g/t'); axes[1].set_ylabel('Frekans (raw)')
        ax2.set_ylabel('Frekans (log)')
        axes[1].set_title('Au Dağılımı — Raw vs Log')
        lines1,labs1=axes[1].get_legend_handles_labels()
        lines2,labs2=ax2.get_legend_handles_labels()
        axes[1].legend(lines1+lines2,labs1+labs2,fontsize=8)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'fig01_data_summary.png'),dpi=150,bbox_inches='tight')
        plt.close(); print("  → figures/fig01_data_summary.png")

    @staticmethod
    def plot_variograms(ev: EmpiricalVariogram):
        fig,axes=plt.subplots(2,3,figsize=(15,9))
        fig.suptitle("3D-VSK — Empirik Variogram ve Model Fitting",
                      fontsize=12,fontweight='bold')
        axes=axes.flatten()
        cols=['#e74c3c','#e67e22','#27ae60','#2980b9']
        h_fine=np.linspace(0.1,70,300)
        for idx,(d,lbl,col) in enumerate(
                zip(ev.DIRECTIONS,ev.DIR_LABELS,cols)):
            ax=axes[idx]
            res=ev.empirical[d]
            ax.scatter(res["h"],res["gamma"],color=col,
                        s=60,edgecolors='k',linewidths=0.8,
                        zorder=5,label='Empirik')
            for hh,gg,np_ in zip(res["h"],res["gamma"],res["n_pairs"]):
                ax.annotate(f'n={np_}',(hh,gg),xytext=(0,6),
                             textcoords='offset points',fontsize=7,
                             ha='center',color='gray')
            p=ev.fitted[d]
            g_m=VariogramModel.spherical_gamma(h_fine,p["C0"],p["C"],p["a"])
            ax.plot(h_fine,g_m,'-',color=col,lw=2,
                     label=f"Spherical C₀={p['C0']:.3f} C={p['C']:.3f} a={p['a']:.1f}m R²={p['R2']:.3f}")
            ax.axhline(p["C0"]+p["C"],color=col,ls='--',lw=1,alpha=0.5)
            ax.axvline(p["a"],color=col,ls=':',lw=1,alpha=0.5)
            ax.set_xlabel('Lag h (m)'); ax.set_ylabel('γ(h)')
            ax.set_title(f'Variogram — {lbl}')
            ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
            ax.set_xlim(0,72); ax.set_ylim(bottom=0)
        # Tüm yönler birlikte
        ax_all=axes[4]
        for d,lbl,col in zip(ev.DIRECTIONS,ev.DIR_LABELS,cols):
            res=ev.empirical[d]; p=ev.fitted[d]
            ax_all.scatter(res["h"],res["gamma"],color=col,s=35,alpha=0.7)
            g_m=VariogramModel.spherical_gamma(h_fine,p["C0"],p["C"],p["a"])
            ax_all.plot(h_fine,g_m,'-',color=col,lw=2,
                         label=f'{lbl} a={p["a"]:.0f}m')
        ax_all.set_xlabel('Lag h (m)'); ax_all.set_ylabel('γ(h)')
        ax_all.set_title('Tüm Yönler — Anizotropi Özeti')
        ax_all.legend(fontsize=7.5); ax_all.grid(True,alpha=0.3)
        # Anizotropi elipsoidi
        ax_ell=axes[5]
        ranges={d:ev.fitted[d]["a"] for d in ev.DIRECTIONS}
        theta_arr=np.linspace(0,360,360)
        # Basit polar: range yöne göre
        a_min_v=min(ranges.values()); a_max_v=max(ranges.values())
        theta_max_v=max(ranges,key=ranges.get)
        a_arr=[a_min_v+(a_max_v-a_min_v)*
               np.cos(np.radians(t%180)-np.radians(theta_max_v))**2
               for t in theta_arr]
        ax_ell.plot(
            np.array(a_arr)*np.cos(np.radians(theta_arr)),
            np.array(a_arr)*np.sin(np.radians(theta_arr)),
            '-',color='#27ae60',lw=2.5)
        for d,r in ranges.items():
            ax_ell.scatter(r*np.cos(np.radians(d)),
                            r*np.sin(np.radians(d)),
                            s=80,color='#e74c3c',zorder=5)
            ax_ell.annotate(f'{d}° a={r:.0f}m',
                             (r*np.cos(np.radians(d)),
                              r*np.sin(np.radians(d))),
                             fontsize=8,xytext=(4,4),
                             textcoords='offset points')
        ax_ell.set_aspect('equal')
        ax_ell.set_xlabel('Range (m)'); ax_ell.set_ylabel('Range (m)')
        ax_ell.set_title('Anizotropi Elipsoidi')
        ax_ell.grid(True,alpha=0.3)
        ax_ell.axhline(0,color='gray',lw=0.5,alpha=0.4)
        ax_ell.axvline(0,color='gray',lw=0.5,alpha=0.4)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'fig02_variograms.png'),dpi=150,bbox_inches='tight')
        plt.close(); print("  → figures/fig02_variograms.png")

    @staticmethod
    def plot_loocv(data, results_list):
        n_meth=len(results_list)
        fig,axes=plt.subplots(2,n_meth,figsize=(5*n_meth,10))
        fig.suptitle("3D-VSK — LOOCV Karşılaştırması",
                      fontsize=13,fontweight='bold')
        au_log=data.au_log; au_raw=np.exp(au_log)
        lim=[au_log.min()-0.15,au_log.max()+0.15]
        lim_bt=[0,2.3]
        cols_m=['#95a5a6','#e67e22','#27ae60','#8e44ad']
        for idx,r in enumerate(results_list):
            col=cols_m[idx%len(cols_m)]
            # Log uzayı scatter
            ax=axes[0,idx]
            sc=ax.scatter(au_log,r["pred"],
                           c=np.abs(r["res"]),cmap='RdYlGn_r',
                           vmin=0,vmax=0.5,s=65,
                           edgecolors='k',linewidths=0.7,zorder=4)
            ax.plot(lim,lim,'k--',lw=1.5,alpha=0.6)
            plt.colorbar(sc,ax=ax,label='|residual|')
            ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')
            ax.set_xlabel('Gerçek ln(Au)'); ax.set_ylabel('Tahmin ln(Au)')
            ax.set_title(f"{r['label']}\nRMSE={r['RMSE']:.4f} R²={r['R2']:.4f}",
                          fontsize=9,fontweight='bold')
            ax.text(0.05,0.92,f"MSDR={r['MSDR']:.3f}",
                     transform=ax.transAxes,fontsize=8,
                     bbox=dict(boxstyle='round',fc='white',alpha=0.8))
            ax.grid(True,alpha=0.3)
            # Back-transform scatter
            ax2=axes[1,idx]
            z_bt=np.exp(r["pred"])
            ax2.scatter(au_raw,z_bt,color=col,s=65,
                         edgecolors='k',linewidths=0.7,zorder=4)
            ax2.plot(lim_bt,lim_bt,'k--',lw=1.5,alpha=0.6)
            ax2.set_xlim(lim_bt); ax2.set_ylim(lim_bt); ax2.set_aspect('equal')
            ax2.set_xlabel('Gerçek Au g/t'); ax2.set_ylabel('Tahmin Au g/t')
            ax2.set_title(f"Back-transform\nRMSE={r['RMSE_bt']:.4f} "
                           f"R²={r['R2_bt']:.4f}",fontsize=9)
            ax2.grid(True,alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'fig03_loocv.png'),dpi=150,bbox_inches='tight')
        plt.close(); print("  → figures/fig03_loocv.png")

    @staticmethod
    def plot_benchmark(results_list, psd_results):
        fig,axes=plt.subplots(1,3,figsize=(16,6))
        fig.suptitle("3D-VSK — Benchmark Özeti: Klasik vs 3D-VSK",
                      fontsize=12,fontweight='bold')
        names =[r["label"] for r in results_list]
        rmses =[r["RMSE"]  for r in results_list]
        r2s   =[r["R2"]    for r in results_list]
        lmins =[p["lmin"]  for p in psd_results]
        cols_b=['#95a5a6','#e67e22','#27ae60','#8e44ad']
        x=np.arange(len(names))
        # RMSE
        bars=axes[0].bar(x,rmses,color=cols_b,edgecolor='k',
                          linewidth=0.8,alpha=0.85)
        for bar,v in zip(bars,rmses):
            axes[0].text(bar.get_x()+bar.get_width()/2,
                          v+0.005,f'{v:.4f}',
                          ha='center',va='bottom',fontsize=9,fontweight='bold')
        axes[0].set_xticks(x); axes[0].set_xticklabels(names,fontsize=9)
        axes[0].set_ylabel('RMSE (log uzayı)'); axes[0].set_title('RMSE Karşılaştırması')
        axes[0].grid(True,alpha=0.3,axis='y')
        # R²
        bars2=axes[1].bar(x,r2s,color=cols_b,edgecolor='k',
                           linewidth=0.8,alpha=0.85)
        for bar,v in zip(bars2,r2s):
            ypos=max(v,0)+0.01
            axes[1].text(bar.get_x()+bar.get_width()/2,
                          ypos,f'{v:.4f}',
                          ha='center',va='bottom',fontsize=9,fontweight='bold')
        axes[1].set_xticks(x); axes[1].set_xticklabels(names,fontsize=9)
        axes[1].set_ylabel('R²'); axes[1].set_title('R² Karşılaştırması')
        axes[1].grid(True,alpha=0.3,axis='y')
        # λ_min
        bar_cols_psd=['#27ae60' if v>-1e-10 else '#e74c3c' for v in lmins]
        bars3=axes[2].bar(x,lmins,color=bar_cols_psd,
                           edgecolor='k',linewidth=0.8,alpha=0.85)
        axes[2].axhline(0,color='black',lw=2,ls='--')
        axes[2].fill_between([-0.5,len(names)-0.5],
                              [-0.5,-0.5],[0,0],
                              alpha=0.08,color='red',label='PSD ihlal bölgesi')
        for bar,v in zip(bars3,lmins):
            axes[2].text(bar.get_x()+bar.get_width()/2,
                          v-(0.02 if v<0 else -0.01),
                          f'{v:.4f}',ha='center',
                          va='top' if v<0 else 'bottom',
                          fontsize=9,fontweight='bold',
                          color='#c0392b' if v<-1e-10 else '#1e8449')
        axes[2].set_xticks(x); axes[2].set_xticklabels(names,fontsize=9)
        axes[2].set_ylabel('λ_min'); axes[2].set_title('PSD Analizi (λ_min)')
        axes[2].legend(fontsize=8); axes[2].grid(True,alpha=0.3,axis='y')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'fig04_benchmark.png'),dpi=150,bbox_inches='tight')
        plt.close(); print("  → figures/fig04_benchmark.png")


# ═════════════════════════════════════════════════════════════
# ANA PIPELINE
# ═════════════════════════════════════════════════════════════


# 3D-VSK LMR Revision — Full Functional Pipeline

This notebook packages **every methodological step** carried out during the
LMR revision of the 3D-VSK manuscript, in response to the reviewer critique
that (1) the original "LMC-Ellipsoidal" formulation used incorrect LMC
terminology and relied on an unproven-PSD heuristic, and (2) the study was
restricted to a single (spherical) variogram model type.

**Prerequisite:** run the original pipeline's Cell 1 (Kalgoorlie data +
`KalgoorlieData`, `EmpiricalVariogram`, `VariogramModel`, `OrdinaryKriging`,
`PSDAnalysis`) before running the cells below.

**Contents:**
1. New PSD-consistent formulations (4 methods, each with a `kernel` option)
2. Old heuristic (kernel-parametrized reference baseline)
3. Generic classes for a second, independent dataset (Seyitömer)
4. Benchmark utilities (fast LOOCV + one-call full comparison)
5. Visualization utilities (surface/contour/eigenvalue/LOOCV/bar-chart figures)
6. Resource-estimation utilities (CZM-based block model + reserve, both
   single- and dual-variable deposits, with true 3D geometry where available)
7. **Runnable driver — Kalgoorlie** (fully reproducible, public data)
8. **Runnable driver — Seyitömer** (requires the confidential real dataset;
   a synthetic proxy generator is provided separately in the original Cell 2
   for public reproducibility)
9. **Extended synthetic PSD counterexample** (Fig. 1, all 4 new methods)

Each formulation's PSD-consistency is *proven*, not empirically observed —
see the class docstrings below for the specific structural argument in each
case (Bochner's theorem and standard corollaries).

## 1. New PSD-Consistent Covariance Formulations

In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════
NEW PSD-CONSISTENT COVARIANCE FORMULATIONS (LMR Revision)
═══════════════════════════════════════════════════════════════════════════
Developed in response to reviewer critique that the original "LMC-Ellipsoidal"
formulation (i) used incorrect LMC terminology for a univariate problem where
LMR (Linear Model of Regionalization) is the appropriate framework, and (ii)
relied on a heuristic anisotropy formula whose PSD-consistency was only
empirically observed, never proven.

Four formulations are provided, each with a genuine, provable PSD guarantee
(not empirical), and each accepting a `kernel` argument ('spherical',
'exponential', or 'gaussian') so that model-type sensitivity can be tested
without changing the surrounding fitting/benchmarking code.

    1. TrueEllipticalAnisotropic  -- corrected single-structure geometric
       anisotropy (the direct fix for the original Eq. 5 heuristic).
    2. ContinuousZonalMixture (CZM) -- first practical fitting procedure for
       the directional-mixture representation of Allard, Senoussi & Porcu
       (2016, Math. Geosci., Theorem 4), which the original authors left as
       an open research question.
    3. NestedLMR -- classical two-structure Linear Model of Regionalization,
       reparametrized (b_k = exp(u_k)) for unconstrained, robust fitting.
    4. KernelSumMixture -- Allard et al.'s own Section 5.2.1 finite
       kernel-sum parametrization of the directional weight function,
       implemented here as a baseline/comparison (not a novel contribution
       of this study).

PSD GUARANTEE (all four formulations): a valid (authorized) isotropic
covariance function composed with a linear coordinate transform, a
non-negative-weighted sum of such functions, or a non-negative-weighted
integral of such functions, is provably PSD (Bochner's theorem and its
standard corollaries; Journel & Huijbregts, 1978; Chilès & Delfiner, 2012).
This is a structural guarantee, not an empirical observation -- and it holds
for any of the three supported kernels.
"""

import numpy as np
from scipy.optimize import least_squares
import warnings
warnings.filterwarnings('ignore')


def _kernel_unit(r, kernel):
    """Unit-sill, unit-range isotropic correlation function, 1 - gamma(r).
    r is the distance normalized by the (possibly direction-dependent) range.
    All three options are authorized (positive-definite) covariance models."""
    r = np.atleast_1d(np.asarray(r, dtype=float))
    if kernel == 'spherical':
        return np.where(r >= 1.0, 1.0, 1.5 * r - 0.5 * r ** 3)
    elif kernel == 'exponential':
        return 1.0 - np.exp(-3.0 * r)          # practical range convention
    elif kernel == 'gaussian':
        return 1.0 - np.exp(-3.0 * r ** 2)     # practical range convention
    raise ValueError(f"Unknown kernel '{kernel}' (use 'spherical'/'exponential'/'gaussian')")


# ═════════════════════════════════════════════════════════════════════════
# 1. TrueEllipticalAnisotropic  ("3D-VSK-Elliptical")
# ═════════════════════════════════════════════════════════════════════════
class TrueEllipticalAnisotropic:
    """
    Corrected replacement for the original Eq. 5 heuristic
    (a(theta) = a_min + (a_max-a_min)*cos^2(theta-theta_max)), which is NOT
    the standard ellipse equation and whose PSD-consistency was therefore
    only empirically observed. This class instead uses the textbook
    geometric-anisotropy transform:

        1/a(theta)^2 = cos^2(theta-theta_max)/a_max^2 + sin^2(theta-theta_max)/a_min^2

    implemented as a genuine coordinate rotation + rescaling, i.e. a
    Mahalanobis-type distance is computed and fed into a single isotropic
    kernel. This composition is PSD by construction for any valid isotropic
    kernel (Journel & Huijbregts, 1978; Chilès & Delfiner, 2012, Sec. 2.5.2).

    Unlike the original per-direction-then-heuristic-ellipse construction,
    all 5 parameters (C0, C, a_max, a_min, theta_max) are fit JOINTLY via
    weighted least squares against all directional empirical variograms
    simultaneously -- addressing the reviewer's related concern about
    directional models being fit independently and reconciled post hoc.

    LIMITATION: a single sill (no zonal/directional-sill anisotropy). See
    NestedLMR for a formulation that adds this capability.
    """

    def __init__(self, ev, kernel='spherical'):
        self.ev = ev
        self.kernel = kernel
        self.n_params = 5
        self.params = None
        self.fit_info = {}

    @staticmethod
    def _r_eff(dx, dy, a_max, a_min, theta_max_deg):
        th = np.radians(theta_max_deg)
        xr = dx * np.cos(th) + dy * np.sin(th)
        yr = -dx * np.sin(th) + dy * np.cos(th)
        return np.sqrt((xr / a_max) ** 2 + (yr / a_min) ** 2)

    def gamma_model(self, h, theta_deg, params):
        h = np.atleast_1d(np.asarray(h, dtype=float))
        C0, C, a_max, a_min, theta_max = params
        th_rad = np.radians(theta_deg)
        dx, dy = h * np.cos(th_rad), h * np.sin(th_rad)
        r = self._r_eff(dx, dy, a_max, a_min, theta_max)
        g = _kernel_unit(r, self.kernel)
        out = np.where(h <= 0, 0.0, C0 + C * g)
        return out if out.size > 1 else float(out[0])

    def cov_func(self, xi, xj, params=None):
        if params is None:
            params = self.params
        if np.allclose(xi, xj):
            return self.cov_diagonal(params)
        C0, C, a_max, a_min, theta_max = params
        dx, dy = xj[0] - xi[0], xj[1] - xi[1]
        r = self._r_eff(dx, dy, a_max, a_min, theta_max)
        g = _kernel_unit(r, self.kernel)
        return float((C - C * g).item())

    def cov_diagonal(self, params=None):
        if params is None:
            params = self.params
        C0, C = params[0], params[1]
        return float(C0 + C)

    def _residuals(self, params):
        res = []
        for d in self.ev.DIRECTIONS:
            emp = self.ev.empirical[d]
            h_d, g_d, np_d = emp["h"], emp["gamma"], emp["n_pairs"]
            if len(h_d) == 0:
                continue
            g_pred = self.gamma_model(h_d, d, params)
            w = np.sqrt(np_d) / np.sqrt(np_d.sum())
            res.append(w * (g_d - g_pred))
        return np.concatenate(res)

    def fit(self, n_restarts=30, seed=0):
        sill_data = np.var(self.ev.data.au_log) if hasattr(self.ev.data, 'au_log') else 1.0
        fitted_ranges = [self.ev.fitted[d]["a"] for d in self.ev.DIRECTIONS if d in self.ev.fitted]
        range_scale = 2.0 * max(fitted_ranges) if fitted_ranges else 200.0
        rng = np.random.default_rng(seed)
        best, all_results = None, []
        for _ in range(n_restarts):
            x0 = np.array([
                0.05 * sill_data * (0.5 + rng.random()),
                0.9 * sill_data * (0.5 + rng.random()),
                0.5 * range_scale * (0.5 + rng.random()),
                0.3 * range_scale * (0.5 + rng.random()),
                180.0 * rng.random(),
            ])
            lb = [1e-6, 1e-6, 1.0, 1.0, 0.0]
            ub = [sill_data, sill_data * 2, range_scale, range_scale, 180.0]
            try:
                sol = least_squares(self._residuals, x0, bounds=(lb, ub), max_nfev=600,
                                     ftol=1e-6, xtol=1e-6, gtol=1e-8)
                cost = float(2 * sol.cost)
                all_results.append({"params": sol.x.copy(), "cost": cost, "success": sol.success})
                if best is None or cost < best["cost"]:
                    best = {"params": sol.x.copy(), "cost": cost}
            except Exception:
                all_results.append({"params": None, "cost": np.inf, "success": False})

        self.params = best["params"]
        C0, C, a1, a2, th = self.params
        if a2 > a1:  # canonical form: a_max >= a_min
            self.params = np.array([C0, C, a2, a1, (th + 90.0) % 180.0])

        n_success = sum(1 for r in all_results if r.get("success"))
        costs = [r["cost"] for r in all_results if r["success"]]
        self.fit_info = {
            "n_restarts": n_restarts,
            "n_converged": n_success,
            "convergence_rate": n_success / n_restarts,
            "cost_std": float(np.std(costs)) if len(costs) > 1 else 0.0,
            "best_cost": best["cost"],
        }
        return self

    def aicc(self):
        n_obs = sum(len(self.ev.empirical[d]["h"]) for d in self.ev.DIRECTIONS)
        k = self.n_params
        rss = float((self._residuals(self.params) ** 2).sum())
        aic = n_obs * np.log(rss / n_obs + 1e-12) + 2 * k
        if n_obs - k - 1 > 0:
            return aic + (2 * k * (k + 1)) / (n_obs - k - 1)
        return np.inf

    def summary(self):
        C0, C, a_max, a_min, th = self.params
        print(f"  kernel={self.kernel}  C0={C0:.5f}  C={C:.5f}  a_max={a_max:.2f}  "
              f"a_min={a_min:.2f}  anisotropy_ratio={a_max/a_min:.2f}  theta_max={th:.1f} deg")


# ═════════════════════════════════════════════════════════════════════════
# 2. ContinuousZonalMixture ("CZM") -- our own methodological contribution
# ═════════════════════════════════════════════════════════════════════════
_GL_NODES, _GL_WEIGHTS = np.polynomial.legendre.leggauss(24)
_PHI_NODES = (np.pi / 2.0) * (_GL_NODES + 1.0)      # 24-point Gauss-Legendre on [0, pi]
_PHI_WEIGHTS = (np.pi / 2.0) * _GL_WEIGHTS


class ContinuousZonalMixture:
    """
    C(h,theta) = c0*1[h=0] + c_iso*Gamma_iso(|h|; a_iso)
                 + integral_0^pi w(phi) * g(h*cos(theta-phi); b) dphi

    This uses the GENERAL mathematical framework of Allard, Senoussi & Porcu
    (2016, Math. Geosci., Theorem 4) -- a directional-mixture representation
    that is PSD by construction whenever w(phi) >= 0. The original authors
    proved this integral is PSD but explicitly left practical parameter
    estimation as an open research question.

    OUR CONTRIBUTION is the specific, closed-form, always-non-negative
    parametrization of w(phi):

        w(phi) = A * exp( sum_k alpha_k*cos(2k*phi) + beta_k*sin(2k*phi) )

    (an exponentiated Fourier series -- a technique familiar from circular/
    directional statistics, applied here for the first time to geostatistical
    zonal-anisotropy mixture weights), together with AICc-based automatic
    selection of the Fourier order K, and validation against real ore data.
    This is the first practical fitting procedure for Allard et al.'s
    framework.

    `kernel` controls BOTH the isotropic background AND the 1D projection
    kernel used inside the zonal integral.
    """

    def __init__(self, ev, K=1, kernel='spherical'):
        self.ev = ev
        self.K = K
        self.kernel = kernel
        self.n_params = 5 + 2 * K
        self.params = None
        self.fit_info = {}

    def w_phi(self, phi, params):
        """Non-negative directional weight density. A > 0 and exp(.) > 0
        always, so w(phi) >= 0 for ANY real-valued Fourier coefficients --
        this is what removes the need for constrained optimization."""
        A = params[4]
        s = np.zeros_like(np.asarray(phi, dtype=float))
        for k in range(1, self.K + 1):
            ak = params[5 + 2 * (k - 1)]
            bk = params[5 + 2 * (k - 1) + 1]
            s = s + ak * np.cos(2 * k * phi) + bk * np.sin(2 * k * phi)
        return A * np.exp(s)

    def zonal_cov(self, h, theta_deg, params):
        b = params[3]
        theta = np.radians(theta_deg)
        proj = h * np.cos(theta - _PHI_NODES)
        gvals = _kernel_unit(np.abs(proj) / b, self.kernel)   # correlation, not variogram
        cov_vals = 1.0 - gvals
        wvals = self.w_phi(_PHI_NODES, params)
        return float(np.sum(_PHI_WEIGHTS * wvals * cov_vals))

    def zonal_sill(self, theta_deg, params):
        return self.zonal_cov(0.0, theta_deg, params)

    def gamma_model(self, h, theta_deg, params):
        c0, c_iso, a_iso = params[0], params[1], params[2]
        h_arr = np.atleast_1d(np.asarray(h, dtype=float))
        r_iso = h_arr / a_iso
        cov_iso = np.where(h_arr <= 0, c_iso, c_iso * (1.0 - _kernel_unit(r_iso, self.kernel)))
        cov_zon_h = np.array([self.zonal_cov(hh, theta_deg, params) for hh in h_arr])
        cov_zon_0 = self.zonal_sill(theta_deg, params)
        total_sill = c0 + c_iso + cov_zon_0
        gamma = total_sill - (cov_iso + cov_zon_h)
        return gamma if h_arr.size > 1 else float(gamma[0])

    def cov_func(self, xi, xj, params=None):
        if params is None:
            params = self.params
        if np.allclose(xi, xj):
            return self.cov_diagonal(params)
        dx, dy = xj[0] - xi[0], xj[1] - xi[1]
        h_h = np.hypot(dx, dy)
        az = np.degrees(np.arctan2(dy, dx)) % 180
        c_iso, a_iso = params[1], params[2]
        cov_iso = c_iso * (1.0 - _kernel_unit(np.array([h_h / a_iso]), self.kernel)[0])
        cov_zon = self.zonal_cov(h_h, az, params)
        return float(cov_iso + cov_zon)

    def cov_diagonal(self, params=None):
        if params is None:
            params = self.params
        c0, c_iso = params[0], params[1]
        return float(c0 + c_iso + self.zonal_sill(0.0, params))

    def _residuals(self, params, lambda_ridge=0.0):
        res = []
        for d in self.ev.DIRECTIONS:
            emp = self.ev.empirical[d]
            h_d, g_d, np_d = emp["h"], emp["gamma"], emp["n_pairs"]
            if len(h_d) == 0:
                continue
            g_pred = self.gamma_model(h_d, d, params)
            w = np.sqrt(np_d) / np.sqrt(np_d.sum())
            res.append(w * (g_d - g_pred))
        res = np.concatenate(res)
        if lambda_ridge > 0.0:
            fourier_coeffs = params[5:5 + 2 * self.K]
            res = np.concatenate([res, np.sqrt(lambda_ridge) * fourier_coeffs])
        return res

    def fit(self, n_restarts=20, seed=0, bound=5.0, lambda_ridge=0.0):
        sill_data = np.var(self.ev.data.au_log) if hasattr(self.ev.data, 'au_log') else 1.0
        fitted_ranges = [self.ev.fitted[d]["a"] for d in self.ev.DIRECTIONS if d in self.ev.fitted]
        range_scale = 2.0 * max(fitted_ranges) if fitted_ranges else 200.0
        rng = np.random.default_rng(seed)
        best = None
        for _ in range(n_restarts):
            x0 = [0.05 * sill_data * (0.5 + rng.random()), 0.3 * sill_data * (0.5 + rng.random()),
                  0.3 * range_scale * (0.5 + rng.random()), 0.3 * range_scale * (0.5 + rng.random()),
                  0.5 * sill_data * (0.3 + rng.random())]
            for _k in range(self.K):
                x0 += [rng.normal(0, 0.5), rng.normal(0, 0.5)]
            x0 = np.array(x0)
            lb = [1e-6, 1e-6, 1.0, 1.0, 1e-6] + [-bound, -bound] * self.K
            ub = [sill_data, sill_data * 2, range_scale, range_scale, sill_data * 3] + [bound, bound] * self.K
            try:
                sol = least_squares(lambda p: self._residuals(p, lambda_ridge), x0,
                                     bounds=(lb, ub), max_nfev=600, ftol=1e-6, xtol=1e-6, gtol=1e-8)
                cost = float(2 * sol.cost)
                if best is None or cost < best[0]:
                    best = (cost, sol.x.copy())
            except Exception:
                pass
        self.params = best[1]
        return self

    def aicc(self):
        n_obs = sum(len(self.ev.empirical[d]["h"]) for d in self.ev.DIRECTIONS)
        k = self.n_params
        rss = float((self._residuals(self.params) ** 2).sum())
        aic = n_obs * np.log(rss / n_obs + 1e-12) + 2 * k
        if n_obs - k - 1 > 0:
            return aic + (2 * k * (k + 1)) / (n_obs - k - 1)
        return np.inf


# ═════════════════════════════════════════════════════════════════════════
# 3. NestedLMR -- unconstrained reparametrization of the classical LMR
# ═════════════════════════════════════════════════════════════════════════
class NestedLMR:
    """
    C(h,theta) = b1*C_1(h,theta) + b2*C_2(h,theta),   b1, b2 >= 0

    This IS the classical, decades-old "gold standard" approach used by
    mining-geostatistics software (GSLIB, Isatis, Vulcan, etc.): a
    non-negative-weighted sum of authorized nested structures. b_k >= 0
    guarantees PSD by the convex-cone argument (Journel & Huijbregts, 1978;
    Wackernagel, 2003).

    Structure 1 (range/geometric anisotropy): the same TrueEllipticalAnisotropic
    construction as above (rotation + rescaling into a single kernel).
    Structure 2 (zonal/sill anisotropy): the classical "practically-infinite
    range" trick -- range -> very large along one axis (theta_z), finite
    range a_z perpendicular to it -- which produces a direction-dependent
    SILL rather than a direction-dependent range.

    OUR CONTRIBUTION: the classical constrained fit (b_k >= 0) is known to
    lack convergence guarantees (Goulard & Voltz, 1992). We reparametrize
    b_k = exp(u_k), making the optimization fully UNCONSTRAINED while
    preserving the non-negativity (hence PSD) guarantee exactly.
    """

    def __init__(self, ev, kernel='spherical', inf_multiplier=50.0):
        self.ev = ev
        self.kernel = kernel
        self.n_params = 8
        self.params = None
        self.fit_info = {}
        self.inf_multiplier = inf_multiplier
        self._a_max_z = None

    @staticmethod
    def _r_eff(dx, dy, a_max, a_min, theta_max_deg):
        th = np.radians(theta_max_deg)
        xr = dx * np.cos(th) + dy * np.sin(th)
        yr = -dx * np.sin(th) + dy * np.cos(th)
        return np.sqrt((xr / a_max) ** 2 + (yr / a_min) ** 2)

    def gamma_model(self, h, theta_deg, params):
        h = np.atleast_1d(np.asarray(h, dtype=float))
        C0, u1, a_max1, a_min1, th1, u2, a_z, th_z = params
        b1, b2 = np.exp(u1), np.exp(u2)
        th_rad = np.radians(theta_deg)
        dx, dy = h * np.cos(th_rad), h * np.sin(th_rad)
        r1 = self._r_eff(dx, dy, a_max1, a_min1, th1)
        g1 = _kernel_unit(r1, self.kernel)
        r2 = self._r_eff(dx, dy, self._a_max_z, a_z, th_z)
        g2 = _kernel_unit(r2, self.kernel)
        gamma = C0 * (h > 0) + b1 * g1 + b2 * g2
        out = np.where(h <= 0, 0.0, gamma)
        return out if out.size > 1 else float(out[0])

    def cov_func(self, xi, xj, params=None):
        if params is None:
            params = self.params
        if np.allclose(xi, xj):
            return self.cov_diagonal(params)
        C0, u1, a_max1, a_min1, th1, u2, a_z, th_z = params
        b1, b2 = np.exp(u1), np.exp(u2)
        dx, dy = xj[0] - xi[0], xj[1] - xi[1]
        r1 = self._r_eff(dx, dy, a_max1, a_min1, th1)
        g1 = _kernel_unit(r1, self.kernel)
        r2 = self._r_eff(dx, dy, self._a_max_z, a_z, th_z)
        g2 = _kernel_unit(r2, self.kernel)
        cov = b1 * (1.0 - g1) + b2 * (1.0 - g2)
        return float(cov.item())

    def cov_diagonal(self, params=None):
        if params is None:
            params = self.params
        C0, u1, _, _, _, u2, _, _ = params
        return float(C0 + np.exp(u1) + np.exp(u2))

    def _residuals(self, params):
        res = []
        for d in self.ev.DIRECTIONS:
            emp = self.ev.empirical[d]
            h_d, g_d, np_d = emp["h"], emp["gamma"], emp["n_pairs"]
            if len(h_d) == 0:
                continue
            g_pred = self.gamma_model(h_d, d, params)
            w = np.sqrt(np_d) / np.sqrt(np_d.sum())
            res.append(w * (g_d - g_pred))
        return np.concatenate(res)

    def fit(self, n_restarts=30, seed=0):
        sill_data = np.var(self.ev.data.au_log) if hasattr(self.ev.data, 'au_log') else 1.0
        fitted_ranges = [self.ev.fitted[d]["a"] for d in self.ev.DIRECTIONS if d in self.ev.fitted]
        range_scale = 2.0 * max(fitted_ranges) if fitted_ranges else 200.0
        self._a_max_z = self.inf_multiplier * range_scale
        rng = np.random.default_rng(seed)
        best, all_results = None, []
        for _ in range(n_restarts):
            x0 = np.array([
                0.05 * sill_data * (0.5 + rng.random()),
                np.log(0.5 * sill_data * (0.3 + rng.random())),
                0.5 * range_scale * (0.5 + rng.random()),
                0.3 * range_scale * (0.5 + rng.random()),
                180.0 * rng.random(),
                np.log(0.3 * sill_data * (0.3 + rng.random())),
                0.3 * range_scale * (0.5 + rng.random()),
                180.0 * rng.random(),
            ])
            lb = [1e-6, -15, 1.0, 1.0, 0.0, -15, 1.0, 0.0]
            ub = [sill_data, np.log(sill_data * 3), range_scale, range_scale, 180.0,
                  np.log(sill_data * 3), range_scale, 180.0]
            try:
                sol = least_squares(self._residuals, x0, bounds=(lb, ub), max_nfev=600,
                                     ftol=1e-6, xtol=1e-6, gtol=1e-8)
                cost = float(2 * sol.cost)
                all_results.append({"params": sol.x.copy(), "cost": cost, "success": sol.success})
                if best is None or cost < best["cost"]:
                    best = {"params": sol.x.copy(), "cost": cost}
            except Exception:
                all_results.append({"params": None, "cost": np.inf, "success": False})

        self.params = best["params"]
        n_success = sum(1 for r in all_results if r.get("success"))
        costs = [r["cost"] for r in all_results if r["success"]]
        self.fit_info = {
            "n_restarts": n_restarts,
            "n_converged": n_success,
            "convergence_rate": n_success / n_restarts,
            "cost_std": float(np.std(costs)) if len(costs) > 1 else 0.0,
            "best_cost": best["cost"],
        }
        return self

    def aicc(self):
        n_obs = sum(len(self.ev.empirical[d]["h"]) for d in self.ev.DIRECTIONS)
        k = self.n_params
        rss = float((self._residuals(self.params) ** 2).sum())
        aic = n_obs * np.log(rss / n_obs + 1e-12) + 2 * k
        if n_obs - k - 1 > 0:
            return aic + (2 * k * (k + 1)) / (n_obs - k - 1)
        return np.inf

    def summary(self):
        C0, u1, a_max1, a_min1, th1, u2, a_z, th_z = self.params
        b1, b2 = np.exp(u1), np.exp(u2)
        print(f"  kernel={self.kernel}  C0={C0:.5f}")
        print(f"  Structure 1 (range anisotropy): b1={b1:.5f}  a_max={a_max1:.2f}  "
              f"a_min={a_min1:.2f}  ratio={a_max1/a_min1:.2f}  theta_max={th1:.1f} deg")
        print(f"  Structure 2 (zonal/sill)      : b2={b2:.5f}  a_z={a_z:.2f}  "
              f"theta_z={th_z:.1f} deg  (sill ratio ~ {(b1+b2)/b1:.2f}x along theta_z)")


# ═════════════════════════════════════════════════════════════════════════
# 4. KernelSumMixture -- Allard et al.'s OWN Section 5.2.1 proposal
#    (baseline / comparison; NOT a novel contribution of this study)
# ═════════════════════════════════════════════════════════════════════════
class KernelSumMixture:
    """
    Implements the finite kernel-sum parametrization of w(phi) that Allard,
    Senoussi & Porcu (2016) themselves suggest in Section 5.2.1, as an
    alternative basis to CZM's exponentiated-Fourier w(phi). We provide the
    first practical fitting/testing of THEIR suggestion; this class exists
    for fair comparison against CZM, not as a contribution we claim.

        w(phi) = sum_{j=1}^{J} exp(u_j) * max(cos(phi-mu_j), 0)^(2*kappa)

    mu_j are J fixed, equally-spaced anchor directions (not fitted). kappa
    controls the shared concentration of the kernels. Non-negativity of
    w(phi) is automatic (squared-cosine kernel, clipped at zero, times a
    positive amplitude).
    """

    def __init__(self, ev, J=4, kernel='spherical'):
        self.ev = ev
        self.J = J
        self.kernel = kernel
        self.mu = np.array([j * np.pi / J for j in range(J)])
        self.n_params = 5 + J
        self.params = None
        self.fit_info = {}

    def w_phi(self, phi, params):
        kappa = params[4]
        u = params[5:5 + self.J]
        w = np.zeros_like(np.asarray(phi, dtype=float))
        for j in range(self.J):
            base = np.cos(phi - self.mu[j])
            base = np.where(base > 0, base, 0.0)
            w = w + np.exp(u[j]) * base ** (2.0 * kappa)
        return w

    def zonal_cov(self, h, theta_deg, params):
        b = params[3]
        theta = np.radians(theta_deg)
        proj = h * np.cos(theta - _PHI_NODES)
        cov_vals = _kernel_unit(np.abs(proj) / b, self.kernel)
        cov_vals = 1.0 - cov_vals
        wvals = self.w_phi(_PHI_NODES, params)
        return float(np.sum(_PHI_WEIGHTS * wvals * cov_vals))

    def zonal_sill(self, theta_deg, params):
        return self.zonal_cov(0.0, theta_deg, params)

    def gamma_model(self, h, theta_deg, params):
        c0, c_iso, a_iso = params[0], params[1], params[2]
        h_arr = np.atleast_1d(np.asarray(h, dtype=float))
        r_iso = h_arr / a_iso
        cov_iso = np.where(h_arr <= 0, c_iso, c_iso * (1.0 - _kernel_unit(r_iso, self.kernel)))
        cov_zon_h = np.array([self.zonal_cov(hh, theta_deg, params) for hh in h_arr])
        cov_zon_0 = self.zonal_sill(theta_deg, params)
        total_sill = c0 + c_iso + cov_zon_0
        gamma = total_sill - (cov_iso + cov_zon_h)
        return gamma if h_arr.size > 1 else float(gamma[0])

    def cov_func(self, xi, xj, params=None):
        if params is None:
            params = self.params
        if np.allclose(xi, xj):
            return self.cov_diagonal(params)
        dx, dy = xj[0] - xi[0], xj[1] - xi[1]
        h_h = np.hypot(dx, dy)
        az = np.degrees(np.arctan2(dy, dx)) % 180
        c_iso, a_iso = params[1], params[2]
        cov_iso = c_iso * (1.0 - _kernel_unit(np.array([h_h / a_iso]), self.kernel)[0])
        cov_zon = self.zonal_cov(h_h, az, params)
        return float(cov_iso + cov_zon)

    def cov_diagonal(self, params=None):
        if params is None:
            params = self.params
        c0, c_iso = params[0], params[1]
        return float(c0 + c_iso + self.zonal_sill(0.0, params))

    def _residuals(self, params):
        res = []
        for d in self.ev.DIRECTIONS:
            emp = self.ev.empirical[d]
            h_d, g_d, np_d = emp["h"], emp["gamma"], emp["n_pairs"]
            if len(h_d) == 0:
                continue
            g_pred = np.array([self.gamma_model(hh, d, params) for hh in h_d])
            w = np.sqrt(np_d) / np.sqrt(np_d.sum())
            res.append(w * (g_d - g_pred))
        return np.concatenate(res)

    def fit(self, n_restarts=20, seed=0, kappa_max=8.0):
        sill_data = np.var(self.ev.data.au_log) if hasattr(self.ev.data, 'au_log') else 1.0
        fitted_ranges = [self.ev.fitted[d]["a"] for d in self.ev.DIRECTIONS if d in self.ev.fitted]
        range_scale = 4.0 * max(fitted_ranges) if fitted_ranges else 200.0
        rng = np.random.default_rng(seed)
        best = None
        for _ in range(n_restarts):
            x0 = np.array([
                0.05 * sill_data * (0.5 + rng.random()),
                0.3 * sill_data * (0.5 + rng.random()),
                0.3 * range_scale * (0.5 + rng.random()),
                0.3 * range_scale * (0.5 + rng.random()),
                1.0 + kappa_max * 0.3 * rng.random(),
            ] + list(np.log(0.2 * sill_data * (0.2 + rng.random(self.J)))))
            lb = [1e-6, 1e-6, 1.0, 1.0, 0.0] + [-15] * self.J
            ub = [sill_data, sill_data * 2, range_scale, range_scale, kappa_max] + [np.log(sill_data * 3)] * self.J
            try:
                sol = least_squares(self._residuals, x0, bounds=(lb, ub), max_nfev=600,
                                     ftol=1e-6, xtol=1e-6, gtol=1e-8)
                cost = float(2 * sol.cost)
                if best is None or cost < best[0]:
                    best = (cost, sol.x.copy())
            except Exception:
                pass
        self.params = best[1]
        return self

    def aicc(self):
        n_obs = sum(len(self.ev.empirical[d]["h"]) for d in self.ev.DIRECTIONS)
        k = self.n_params
        rss = float((self._residuals(self.params) ** 2).sum())
        aic = n_obs * np.log(rss / n_obs + 1e-12) + 2 * k
        if n_obs - k - 1 > 0:
            return aic + (2 * k * (k + 1)) / (n_obs - k - 1)
        return np.inf


print("Cell loaded: TrueEllipticalAnisotropic, ContinuousZonalMixture, NestedLMR, KernelSumMixture "
      "(all support kernel in {'spherical','exponential','gaussian'})")


## 2. Reference Baseline: Old Heuristic (kernel-parametrized)

Included to demonstrate that its PSD-consistency, unlike the four formulations above, is empirical only — and fragile (see the Kalgoorlie benchmark below, where it loses PSD under a Gaussian kernel).

In [ ]:
"""
Kernel-parametrized version of the ORIGINAL "3D-VSK (cos^2)" heuristic
formulation (the manuscript's Eq. 5, `VariogramSurface3D.cov_lmc_ellipsoidal`
in Cell 1), included here purely as a REFERENCE / baseline row in the
benchmark tables below.

PSD-consistency of this formulation was only ever empirically observed
(never proven) -- and the kernel-choice experiment below demonstrates
exactly why that matters: at Kalgoorlie, swapping the base variogram model
from spherical to Gaussian causes this heuristic's covariance matrix to
LOSE positive semi-definiteness (see the benchmark output further down).
This is a key piece of evidence for the reviewer-response argument that
empirically-observed PSD is a fragile substitute for a structural guarantee.
"""

import numpy as np


class OldVSKHeuristic:
    """theta_max/a_min/a_max/nugget/sill parameters are derived directly
    from the per-direction fits (exactly as in the original manuscript),
    NOT re-optimized -- this class only swaps the base kernel used to
    turn the heuristic ellipse range a(theta) into a covariance value."""

    def __init__(self, ev, kernel='spherical'):
        self.ev = ev
        self.kernel = kernel
        nugs = [ev.fitted[d]['C0'] for d in ev.DIRECTIONS]
        sills = [ev.fitted[d]['C'] for d in ev.DIRECTIONS]
        ranges = {d: ev.fitted[d]['a'] for d in ev.DIRECTIONS}
        self.b_nug = np.mean(nugs)
        self.b_str = np.mean(sills)
        self.a_min = min(ranges.values())
        self.a_max = max(ranges.values())
        self.theta_max = max(ranges, key=ranges.get)
        self.n_params = 6  # b_nug, b_str, a_min, a_max, theta_max, a_vert_f (fixed, unused here)

    def a_ellipse(self, theta_deg):
        """The ORIGINAL Eq. 5 heuristic -- NOT the true ellipse equation
        (compare to TrueEllipticalAnisotropic's _r_eff, which uses the
        correct 1/a^2 = cos^2/a_max^2 + sin^2/a_min^2 form)."""
        dt = np.radians(theta_deg - self.theta_max)
        return self.a_min + (self.a_max - self.a_min) * np.cos(dt) ** 2

    def gamma_model(self, h, theta_deg, params=None):
        h = np.atleast_1d(np.asarray(h, dtype=float))
        a_h = self.a_ellipse(theta_deg)
        r = h / a_h
        g = _kernel_unit(r, self.kernel)
        out = np.where(h <= 0, 0.0, self.b_nug + self.b_str * g)
        return out if out.size > 1 else float(out[0])

    def cov_func(self, xi, xj, params=None):
        if np.allclose(xi, xj):
            return self.cov_diagonal()
        dx, dy = xj[0] - xi[0], xj[1] - xi[1]
        h = np.hypot(dx, dy)
        az = np.degrees(np.arctan2(dy, dx)) % 180
        a_h = self.a_ellipse(az)
        r = h / a_h
        return float((self.b_str * (1.0 - _kernel_unit(r, self.kernel))).item())

    def cov_diagonal(self, params=None):
        return float(self.b_nug + self.b_str)

    def aicc(self):
        """Not fit via optimization (parameters are derived post-hoc from
        independent per-direction fits), so AICc is reported for reference
        only and is not directly comparable to the jointly-fit methods'
        AICc on equal footing."""
        n_obs = sum(len(self.ev.empirical[d]["h"]) for d in self.ev.DIRECTIONS)
        k = self.n_params
        res = []
        for d in self.ev.DIRECTIONS:
            emp = self.ev.empirical[d]
            h_d, g_d, np_d = emp["h"], emp["gamma"], emp["n_pairs"]
            if len(h_d) == 0:
                continue
            g_pred = self.gamma_model(h_d, d)
            w = np.sqrt(np_d) / np.sqrt(np_d.sum())
            res.append(w * (g_d - g_pred))
        rss = float((np.concatenate(res) ** 2).sum())
        aic = n_obs * np.log(rss / n_obs + 1e-12) + 2 * k
        if n_obs - k - 1 > 0:
            return aic + (2 * k * (k + 1)) / (n_obs - k - 1)
        return np.inf


print("Cell loaded: OldVSKHeuristic (reference baseline, empirical-PSD-only)")


## 3. Generic Field / Empirical-Variogram Classes (for Seyitömer or any other dataset)

In [ ]:
"""
Generic field/empirical-variogram classes for the Seyitömer-Aslanlı lignite
dataset (or any other dataset with real X, Y, Z coordinates and a scalar
regionalized variable). These generalize the Kalgoorlie-specific
EmpiricalVariogram class from Cell 1 to arbitrary lag settings.
"""

import numpy as np
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')


class GenericFieldData:
    """coords: (n,3) array; values: (n,) array. `au_log` name kept for
    backward compatibility with EmpiricalVariogram/CZM/etc., which were
    originally written for the (log-transformed) Kalgoorlie Au dataset."""
    def __init__(self, coords, values):
        self.coords = coords
        self.au_log = values
        self.n = len(values)


class GenericEmpiricalVariogram(EmpiricalVariogram):
    """Same 4-principal-direction (0/45/90/135 deg) empirical variogram
    machinery as Cell 1's EmpiricalVariogram, generalized to arbitrary lag
    width/count/start and a maximum fitting range (needed because Seyitömer's
    variogram ranges are large relative to Kalgoorlie's)."""
    DIRECTIONS = [0, 45, 90, 135]
    DIR_LABELS = ["0 deg E-W", "45 deg NE-SW", "90 deg N-S", "135 deg NW-SE"]

    def __init__(self, data, angle_tol=22.5, lag_width=200.0, n_lags=8,
                 lag_start=150.0, max_range=1245.0):
        self.data = data
        self.angle_tol = angle_tol
        self.lag_width = lag_width
        self.n_lags = n_lags
        self.lag_start = lag_start
        self.max_range = max_range
        self.empirical = {}
        self.fitted = {}

    def fit_all(self):
        sill_init = np.var(self.data.au_log)
        for d in self.DIRECTIONS:
            res = self.empirical[d]
            h_d, g_d = res["h"], res["gamma"]
            if len(h_d) < 3:
                self.fitted[d] = {"C0": 0.01 * sill_init, "C": sill_init * 0.9,
                                   "a": self.max_range * 0.5, "R2": 0.0}
                continue
            try:
                popt, _ = curve_fit(
                    VariogramModel.spherical_gamma, h_d, g_d,
                    p0=[sill_init * 0.1, sill_init * 0.9, h_d.max() * 0.7],
                    bounds=([0, 1e-6, 1], [sill_init * 2, sill_init * 3, self.max_range]),
                    maxfev=8000)
                C0f, Cf, af = popt
                g_pred = VariogramModel.spherical_gamma(h_d, *popt)
                ss_res = np.sum((g_d - g_pred) ** 2)
                ss_tot = np.sum((g_d - g_d.mean()) ** 2)
                r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
                self.fitted[d] = {"C0": C0f, "C": Cf, "a": af, "R2": r2}
            except Exception:
                self.fitted[d] = {"C0": 0.01 * sill_init, "C": sill_init * 0.9,
                                   "a": self.max_range * 0.5, "R2": 0.0}
        return self


print("Cell loaded: GenericFieldData, GenericEmpiricalVariogram "
      "(for Seyitömer or any other real dataset)")


## 4. Benchmark Utilities

In [ ]:
"""
Benchmark utilities: fast neighbourhood-restricted LOOCV for large datasets,
and a single `run_benchmark()` function that fits all four PSD-consistent
formulations (optionally across all three kernels) and reports PSD status,
AICc, LOOCV R²/RMSE, and condition number in one table.
"""

import numpy as np
from scipy.linalg import solve
import warnings
warnings.filterwarnings('ignore')


class FastNeighborLOOCV:
    """K_pp-nearest-neighbour-restricted LOOCV (consistent with the
    manuscript's Section 3.2/S3.2 search-neighbourhood methodology).
    Reduces the near-O(n^4) cost of full-data LOOCV for large n (e.g.
    Seyitömer, n=191) to O(n * K_pp^3) by restricting the kriging system
    at each left-out point to its K_pp nearest neighbours."""

    def __init__(self, coords, values, cov_func, cov_diag, k_pp=20, search_radius=None):
        from scipy.spatial import KDTree
        self.coords = coords
        self.values = values
        self.cov_func = cov_func
        self.cov_diag = cov_diag
        self.k_pp = k_pp
        self.search_radius = search_radius
        self.tree = KDTree(coords[:, :2])

    def run(self, label=""):
        n = len(self.values)
        preds = np.zeros(n)
        kriging_var = np.zeros(n)
        for i in range(n):
            x0 = self.coords[i]
            k_query = min(self.k_pp + 1, n)
            dists, idx = self.tree.query(x0[:2], k=k_query)
            idx = idx[idx != i][: self.k_pp]
            if self.search_radius is not None:
                d_sub, _ = self.tree.query(x0[:2], k=k_query)
                mask = d_sub[: len(idx) + 1] <= self.search_radius
                idx = idx[mask[1:len(idx) + 1]] if len(mask) > 1 else idx
            m = len(idx)
            if m < 3:
                preds[i] = self.values[idx].mean() if m > 0 else self.values.mean()
                kriging_var[i] = self.cov_diag
                continue
            neigh = self.coords[idx]
            K = np.empty((m, m))
            for a in range(m):
                K[a, a] = self.cov_diag
                for bb in range(a + 1, m):
                    v = self.cov_func(neigh[a], neigh[bb])
                    K[a, bb] = K[bb, a] = v
            k0 = np.array([self.cov_func(neigh[a], x0) for a in range(m)])
            A = np.zeros((m + 1, m + 1))
            A[:m, :m] = K
            A[:m, m] = 1.0
            A[m, :m] = 1.0
            b_vec = np.zeros(m + 1)
            b_vec[:m] = k0
            b_vec[m] = 1.0
            try:
                w = solve(A, b_vec, assume_a="sym")
            except Exception:
                w, *_ = np.linalg.lstsq(A, b_vec, rcond=None)
            weights = w[:m]
            preds[i] = float(weights @ self.values[idx])
            mu = w[m]
            kriging_var[i] = self.cov_diag - float(weights @ k0) - mu

        errors = self.values - preds
        me = float(errors.mean())
        rmse = float(np.sqrt((errors ** 2).mean()))
        ss_res = float((errors ** 2).sum())
        ss_tot = float(((self.values - self.values.mean()) ** 2).sum())
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        msdr = float(np.mean(errors ** 2 / np.clip(kriging_var, 1e-9, None)))
        return {"label": label, "ME": me, "RMSE": rmse, "R2": r2, "MSDR": msdr, "n": n,
                "predictions": preds}


class _CovWrapper:
    """Adapts a bare (cov_func, cov_diagonal_value) pair to the interface
    OrdinaryKriging expects (an object with a .cov_diagonal() method)."""
    def __init__(self, cov_func, diag_value):
        self._f = cov_func
        self._d = diag_value
    def cov_diagonal(self):
        return self._d


def run_benchmark(ev, coords, values, n_restarts=20, k_pp=None, kernels=('spherical', 'exponential', 'gaussian'),
                   include_kernelsum=True, seed=42, verbose=True):
    """
    Fits TrueEllipticalAnisotropic, ContinuousZonalMixture (K=1), NestedLMR,
    and (optionally) KernelSumMixture for each requested kernel, evaluates
    PSD status / condition number / AICc, and runs LOOCV (exact for small n
    via OrdinaryKriging, or K_pp-restricted via FastNeighborLOOCV for large n
    -- pass k_pp to select the latter).

    Returns a list of result dicts, one per (method, kernel) combination,
    each already containing PSD/AICc/R2/RMSE/kappa for direct tabulation.
    """
    results = []

    def _evaluate(model, label):
        Cm = PSDAnalysis.build_matrix(coords, model.cov_func, model.cov_diagonal())
        p = PSDAnalysis.check(Cm, label)
        if k_pp is None:
            ok = OrdinaryKriging(model)
            r = ok.loocv(coords, values, model.cov_func, label)
        else:
            loo = FastNeighborLOOCV(coords, values, model.cov_func, model.cov_diagonal(), k_pp=k_pp)
            r = loo.run(label)
        aicc = model.aicc()
        row = {"method": label, "n_params": model.n_params, "AICc": aicc,
               "lambda_min": p["lmin"], "kappa": p["cond"], "PSD": p["psd"],
               "R2": r["R2"], "RMSE": r["RMSE"]}
        if verbose:
            print(f"  {label:<28} n_par={model.n_params:2d}  AICc={aicc:9.2f}  "
                  f"lambda_min={p['lmin']:12.5g}  kappa={p['cond']:9.1f}  "
                  f"PSD={'Y' if p['psd'] else 'N'}  R2={r['R2']:8.4f}  RMSE={r['RMSE']:9.4f}")
        return row

    for kernel in kernels:
        te = TrueEllipticalAnisotropic(ev, kernel=kernel)
        te.fit(n_restarts=n_restarts, seed=seed)
        results.append(_evaluate(te, f"3D-VSK-Elliptical [{kernel}]"))

    for kernel in kernels:
        lmr = NestedLMR(ev, kernel=kernel)
        lmr.fit(n_restarts=n_restarts, seed=seed)
        results.append(_evaluate(lmr, f"Nested-LMR [{kernel}]"))

    for kernel in kernels:
        czm = ContinuousZonalMixture(ev, K=1, kernel=kernel)
        czm.fit(n_restarts=n_restarts, seed=seed, bound=5.0)
        results.append(_evaluate(czm, f"CZM [{kernel}]"))

    if include_kernelsum:
        for kernel in kernels:
            ksm = KernelSumMixture(ev, J=4, kernel=kernel)
            ksm.fit(n_restarts=max(6, n_restarts // 2), seed=seed)
            results.append(_evaluate(ksm, f"KernelSum [{kernel}]"))

    return results


print("Cell loaded: FastNeighborLOOCV, run_benchmark()")


## 5. Visualization Utilities

In [ ]:
"""
Reusable, parametrized visualization functions for the LMR revision. Each
function takes a dict of {label: fitted_model_object} (models must expose
.cov_func, .cov_diagonal(), and .gamma_model(h, theta_deg, params) where
relevant) plus dataset coordinates/values, and produces a single figure.

Colour convention (consistent across ALL figure types, per method):
    3D-VSK-Elliptical -> orange  '#e67e22'
    CZM               -> blue    '#2980b9'
    Nested-LMR        -> purple  '#8e44ad'
    KernelSum         -> teal    '#16a085'
    Discrete/old-VSK  -> grey    '#7f8c8d'  (reference-only, not PSD-proven)
"""

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

METHOD_COLORS = {
    'Discrete': '#7f8c8d', 'Old-VSK (cos2)': '#7f8c8d',
    '3D-VSK-Elliptical': '#e67e22', 'CZM': '#2980b9',
    'Nested-LMR': '#8e44ad', 'KernelSum': '#16a085',
}


def plot_benchmark_bars(results, methods_order, out_path, rmse_label='RMSE'):
    """results: list of dicts from run_benchmark() (or manually built),
    each with keys 'method','R2','RMSE','kappa'. methods_order: list of
    (label, display_name, color) to plot, in order."""
    names = [d[1] for d in methods_order]
    colors = [d[2] for d in methods_order]
    rows = {r['method']: r for r in results}
    rmses = [rows[d[0]]['RMSE'] for d in methods_order]
    r2s = [rows[d[0]]['R2'] for d in methods_order]
    konds = [rows[d[0]]['kappa'] for d in methods_order]
    x = np.arange(len(names))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
    fig.patch.set_facecolor('white')

    bars = axes[0].bar(x, rmses, color=colors, edgecolor='k', linewidth=0.8, alpha=0.9)
    for bar, v in zip(bars, rmses):
        axes[0].text(bar.get_x() + bar.get_width() / 2, v + 0.02 * max(rmses), f'{v:.4f}',
                     ha='center', va='bottom', fontsize=9, fontweight='bold')
    axes[0].set_xticks(x); axes[0].set_xticklabels(names, fontsize=9)
    axes[0].set_ylabel(rmse_label, fontsize=10)
    axes[0].set_title('LOOCV RMSE', fontsize=10, fontweight='bold')
    axes[0].set_ylim(0, max(rmses) * 1.2)
    axes[0].grid(True, alpha=0.3, axis='y')

    bars2 = axes[1].bar(x, r2s, color=colors, edgecolor='k', linewidth=0.8, alpha=0.9)
    for bar, v in zip(bars2, r2s):
        axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.012, f'{v:.4f}',
                     ha='center', va='bottom', fontsize=9, fontweight='bold')
    axes[1].set_xticks(x); axes[1].set_xticklabels(names, fontsize=9)
    axes[1].set_ylabel('R2', fontsize=10)
    axes[1].set_title('LOOCV R2', fontsize=10, fontweight='bold')
    axes[1].set_ylim(0, max(r2s) * 1.25)
    axes[1].grid(True, alpha=0.3, axis='y')

    bars3 = axes[2].bar(x, konds, color=colors, edgecolor='k', linewidth=0.8, alpha=0.9)
    axes[2].set_yscale('log')
    axes[2].set_ylim(top=max(konds) * 4)
    for bar, v in zip(bars3, konds):
        axes[2].text(bar.get_x() + bar.get_width() / 2, v * 1.2, f'{v:.0f}',
                     ha='center', va='bottom', fontsize=9, fontweight='bold')
    axes[2].set_xticks(x); axes[2].set_xticklabels(names, fontsize=9)
    axes[2].set_ylabel('kappa(K)  [log scale]', fontsize=10)
    axes[2].set_title('Condition Number (robustness)', fontsize=10, fontweight='bold')
    axes[2].grid(True, alpha=0.3, axis='y', which='both')

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return out_path


def plot_surface_3d(models, h_max, out_path, n_cols=2):
    """models: list of (label, model_object). Renders one 3D surface panel
    per model in a grid with n_cols columns."""
    n = len(models)
    n_rows = int(np.ceil(n / n_cols))
    h_arr = np.linspace(0, h_max, 60)
    ang_arr = np.linspace(0, 180, 72)
    H, A = np.meshgrid(h_arr, ang_arr)
    Zs = {}
    for label, model in models:
        Zs[label] = np.array([[model.gamma_model(h, a, model.params) for h in h_arr] for a in ang_arr])
    vmax_all = max(z.max() for z in Zs.values())

    fig = plt.figure(figsize=(6.5 * n_cols, 5.2 * n_rows))
    fig.patch.set_facecolor('white')
    for idx, (label, model) in enumerate(models):
        ax = fig.add_subplot(n_rows, n_cols, idx + 1, projection='3d')
        surf = ax.plot_surface(H, A, Zs[label], cmap='YlOrRd', alpha=0.85, linewidth=0,
                                antialiased=True, vmin=0, vmax=vmax_all)
        ax.set_xlabel('h', fontsize=7, labelpad=8)
        ax.set_ylabel('theta (deg)', fontsize=7, labelpad=8)
        ax.set_zlabel('gamma(h,theta)', fontsize=7, labelpad=2)
        ax.set_yticks([0, 45, 90, 135, 180])
        ax.set_title(label, fontsize=10, fontweight='bold', pad=8)
        cbar = fig.colorbar(surf, ax=ax, shrink=0.42, pad=0.13, aspect=18)
        cbar.set_label('gamma', fontsize=8)
        ax.view_init(elev=28, azim=-50)
        ax.tick_params(axis='both', labelsize=7)
    fig.subplots_adjust(wspace=0.35, hspace=0.25, left=0.02, right=0.98, top=0.95, bottom=0.03)
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return out_path


def plot_contour_cross_section(models, h_max, out_path, h_cut_fracs=(0.15, 0.35, 0.65)):
    """models: list of (label, model_object). One ROW per model: left=contour,
    right=directional cross-section at three lag distances."""
    n = len(models)
    h_arr = np.linspace(0, h_max, 60)
    ang_arr = np.linspace(0, 180, 72)
    H, A = np.meshgrid(h_arr, ang_arr)
    Zs = {}
    for label, model in models:
        Zs[label] = np.array([[model.gamma_model(h, a, model.params) for h in h_arr] for a in ang_arr])
    vmax_all = max(z.max() for z in Zs.values())
    ang_fine = np.linspace(0, 180, 360)
    h_cuts = [round(h_max * f) for f in h_cut_fracs]
    hcut_cols = ['#3498db', '#e74c3c', '#2ecc71']

    fig, axes = plt.subplots(n, 2, figsize=(11, 4.1 * n))
    fig.patch.set_facecolor('white')
    if n == 1:
        axes = axes.reshape(1, 2)
    for row, (label, model) in enumerate(models):
        ax1 = axes[row, 0]
        cf = ax1.contourf(H, A, Zs[label], levels=20, cmap='YlOrRd', vmin=0, vmax=vmax_all)
        for d in [0, 45, 90, 135]:
            ax1.axhline(d, color='navy', lw=1.3, ls='--', alpha=0.6)
        ax1.set_yticks([0, 45, 90, 135, 180])
        ax1.set_xlabel('Lag h', fontsize=8); ax1.set_ylabel('Direction theta (deg)', fontsize=8)
        ax1.set_title(f'{label} - contour', fontsize=9, fontweight='bold')
        plt.colorbar(cf, ax=ax1, label='gamma(h,theta)')
        ax1.grid(True, alpha=0.2); ax1.tick_params(labelsize=7)

        ax2 = axes[row, 1]
        for h_f, hcol in zip(h_cuts, hcut_cols):
            vals = [model.gamma_model(h_f, a, model.params) for a in ang_fine]
            ax2.plot(ang_fine, vals, '-', color=hcol, lw=2.0, label=f'h={h_f}')
        for d in [0, 45, 90, 135]:
            ax2.axvline(d, color='gray', ls=':', lw=1, alpha=0.5)
        ax2.set_xlabel('Direction theta (deg)', fontsize=8); ax2.set_ylabel('gamma(h,theta)', fontsize=8)
        ax2.set_xticks([0, 45, 90, 135, 180])
        ax2.set_title(f'{label} - directional cross-section', fontsize=9, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.set_xlim(0, 180); ax2.set_ylim(0, vmax_all * 1.1)
        ax2.legend(fontsize=7.5, loc='upper right'); ax2.tick_params(labelsize=7)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return out_path


def plot_eigenvalue_spectrum(models_with_cov, coords, out_path, n_sub=80):
    """models_with_cov: list of (label, cov_func, cov_diag_value). Subsamples
    to n_sub points for tractable eigenvalue decomposition on large datasets."""
    idx_sub = np.linspace(0, len(coords) - 1, min(n_sub, len(coords))).astype(int)
    coords_sub = coords[idx_sub]
    n = len(models_with_cov)
    n_cols = 2
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(11, 4.3 * n_rows))
    fig.patch.set_facecolor('white')
    axes = np.atleast_2d(axes)
    for idx, (label, cov_func, diag) in enumerate(models_with_cov):
        Cm = np.array([[diag if i == j else cov_func(coords_sub[i], coords_sub[j])
                         for j in range(len(coords_sub))] for i in range(len(coords_sub))])
        eigvals = np.sort(np.linalg.eigvalsh(Cm))
        ax = axes[idx // n_cols, idx % n_cols]
        color = METHOD_COLORS.get(label.split(' [')[0], '#34495e')
        bar_colors = ['#e74c3c' if v < 0 else color for v in eigvals]
        ax.bar(np.arange(len(eigvals)), eigvals, color=bar_colors, edgecolor='k', linewidth=0.4, width=0.85)
        ax.axhline(0, color='black', lw=1.2)
        n_neg = int(np.sum(eigvals < -1e-10))
        ax.set_title(f'{label}  (n_neg={n_neg}, lambda_min={eigvals[0]:.4g})', fontsize=10, fontweight='bold')
        ax.set_xlabel('Eigenvalue index (sorted)', fontsize=9)
        ax.set_ylabel('lambda', fontsize=9)
        ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return out_path


def plot_loocv_scatter(models_with_cov, coords, values, out_path, obs_label='Observed value', k_pp=None):
    """models_with_cov: list of (label, cov_func, cov_diag_value).
    Uses exact OrdinaryKriging LOOCV if k_pp is None, else FastNeighborLOOCV."""
    n = len(models_with_cov)
    n_cols = 2
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 5 * n_rows))
    fig.patch.set_facecolor('white')
    axes = np.atleast_2d(axes)
    for idx, (label, cov_func, diag) in enumerate(models_with_cov):
        color = METHOD_COLORS.get(label.split(' [')[0], '#34495e')
        if k_pp is None:
            wrapper = _CovWrapper(cov_func, diag)
            ok = OrdinaryKriging(wrapper)
            preds = []
            for i in range(len(coords)):
                mask = np.ones(len(coords), dtype=bool); mask[i] = False
                pred, _ = ok.predict(coords[i], coords[mask], values[mask], cov_func)
                preds.append(pred)
            preds = np.array(preds)
        else:
            loo = FastNeighborLOOCV(coords, values, cov_func, diag, k_pp=k_pp)
            res = loo.run(label)
            preds = res['predictions']
        obs = values
        ax = axes[idx // n_cols, idx % n_cols]
        ax.scatter(obs, preds, color=color, s=40, edgecolors='k', linewidths=0.6, alpha=0.8, zorder=3)
        lims = [min(obs.min(), preds.min()) - 0.05 * abs(obs.min() + 1e-6),
                max(obs.max(), preds.max()) + 0.05 * abs(obs.max() + 1e-6)]
        ax.plot(lims, lims, 'k--', lw=1.3, alpha=0.7, label='1:1')
        ss_res = np.sum((obs - preds) ** 2); ss_tot = np.sum((obs - obs.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot
        rmse = np.sqrt(np.mean((obs - preds) ** 2))
        ax.set_xlabel(obs_label, fontsize=9); ax.set_ylabel('Predicted', fontsize=9)
        ax.set_title(f'{label}  (R2={r2:.3f}, RMSE={rmse:.3f})', fontsize=10, fontweight='bold')
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.legend(fontsize=8, loc='upper left')
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return out_path


print("Cell loaded: plot_benchmark_bars, plot_surface_3d, plot_contour_cross_section, "
      "plot_eigenvalue_spectrum, plot_loocv_scatter")


## 6. Resource Estimation Utilities (CZM-based block model + reserve)

In [ ]:
"""
CZM-based 3D block modelling and resource/reserve estimation, generalized
for both single-variable (Kalgoorlie Au) and dual-variable (Seyitömer
thickness + calorific value) deposits.

Grid construction uses a DOUBLE filter -- convex hull AND search-radius --
matching the manuscript's own S4.3 methodology (raw grid -> hull-exclude ->
radius-exclude -> active blocks). Using only the convex-hull filter
over-extends the active domain to unsupported peripheral blocks and
inflates tonnage; this was diagnosed and corrected during this revision.
"""

import numpy as np
from scipy.spatial import ConvexHull, cKDTree
from matplotlib.path import Path
import matplotlib.pyplot as plt


def build_block_grid(coords_xy, step, search_radius):
    """Regular grid clipped by (1) the convex hull of the data locations and
    (2) requiring at least one data point within `search_radius`. Returns
    XG, YG (meshgrid arrays) and a boolean `active` mask of the same shape."""
    hull = ConvexHull(coords_xy)
    hull_path = Path(coords_xy[hull.vertices])
    xmin, xmax = coords_xy[:, 0].min(), coords_xy[:, 0].max()
    ymin, ymax = coords_xy[:, 1].min(), coords_xy[:, 1].max()
    xg = np.arange(xmin, xmax + step, step)
    yg = np.arange(ymin, ymax + step, step)
    XG, YG = np.meshgrid(xg, yg)
    pts = np.column_stack([XG.ravel(), YG.ravel()])
    inside_hull = hull_path.contains_points(pts)
    tree = cKDTree(coords_xy)
    has_neighbor = np.array([len(tree.query_ball_point(p, r=search_radius)) > 0 for p in pts])
    active = (inside_hull & has_neighbor).reshape(XG.shape)
    return XG, YG, active, tree


def krige_grid(model, XG, YG, z_level, active, coords3, values, tree, search_radius, min_neighbors=3):
    """Ordinary-krige `model` at every active grid node (using all data
    points within `search_radius`, falling back to the 10 nearest if fewer
    than `min_neighbors` are found), returning (Z_HAT, SIG2) arrays."""
    shape = XG.shape
    Z_HAT = np.full(shape, np.nan)
    SIG2 = np.full(shape, np.nan)
    ok = OrdinaryKriging(model)
    coords_xy = coords3[:, :2]
    idx_active = np.argwhere(active)
    for (i, j) in idx_active:
        p_xy = np.array([XG[i, j], YG[i, j]])
        nbrs = tree.query_ball_point(p_xy, r=search_radius)
        if len(nbrs) < min_neighbors:
            nbrs = tree.query(p_xy, k=10)[1].tolist()
        nb = np.array(nbrs)
        p3 = np.array([XG[i, j], YG[i, j], z_level])
        zh, s2 = ok.predict(p3, coords3[nb], values[nb], model.cov_func)
        Z_HAT[i, j] = zh
        SIG2[i, j] = s2
    return Z_HAT, SIG2


def jensen_correct_log(z_hat, sig2):
    """Back-transform for a log-transformed variable (e.g. ln(Au)),
    correcting for the systematic underestimation of exp(.) at the mean
    (Eq. 9-equivalent for the log case): Z = exp(Y + 0.5*sigma^2)."""
    return np.exp(z_hat + 0.5 * sig2)


def jensen_correct_sqrt(z_hat, sig2):
    """Back-transform for a sqrt-transformed variable (e.g. sqrt(thickness)),
    per the manuscript's Eq. 9: Z = Y^2 + sigma^2."""
    return z_hat ** 2 + sig2


def compute_reserve_single(Z_BT, active, step, density, cutoff):
    """Single-variable reserve (e.g. Kalgoorlie Au grade): returns a dict
    with active block count, mean grade, tonnage, and cut-off resource."""
    vals = Z_BT[active]
    n_active = active.sum()
    block_vol = step * step * 2.0  # 2 m nominal block height, per manuscript
    tonnage_t = n_active * block_vol * density
    mean_grade = np.nanmean(vals)
    oz = tonnage_t * mean_grade / 31.1035
    n_reserve = int(np.sum(vals >= cutoff))
    return {"n_active": int(n_active), "mean_grade": float(mean_grade),
            "tonnage_t": float(tonnage_t), "oz": float(oz), "n_reserve": n_reserve}


def compute_reserve_dual(THICK_BT, CAL_BT, active, step, density, cutoff):
    """Dual-variable reserve (Seyitömer: thickness + calorific value).
    Tonnage = area * thickness * density per block; reserve = blocks with
    calorific >= cutoff."""
    area = step * step
    tonnage_block = area * THICK_BT * density
    total_tonnage_mt = np.nansum(tonnage_block[active]) / 1e6
    mean_thick = np.nanmean(THICK_BT[active])
    mean_cal = np.nanmean(CAL_BT[active])
    reserve_mask = active & (CAL_BT >= cutoff)
    reserve_tonnage_mt = np.nansum(tonnage_block[reserve_mask]) / 1e6
    reserve_mean_cal = np.nanmean(CAL_BT[reserve_mask]) if reserve_mask.any() else np.nan
    return {"n_active": int(active.sum()), "mean_thickness": float(mean_thick),
            "mean_calorific": float(mean_cal), "total_tonnage_mt": float(total_tonnage_mt),
            "n_reserve": int(reserve_mask.sum()), "reserve_tonnage_mt": float(reserve_tonnage_mt),
            "reserve_mean_calorific": float(reserve_mean_cal)}


# ── Contour-map visualizations ──────────────────────────────────────────
def plot_isograde_map(XG, YG, Z_BT, active, coords_xy, cutoff, out_path,
                       value_label='Grade', cmap='YlOrRd'):
    """3-panel classic contour map: (a) iso-value contour, (b) reserve
    classification at `cutoff`, (c) same value re-shown restricted to
    reserve blocks only (for visual emphasis)."""
    Zp = np.where(active, Z_BT, np.nan)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    fig.patch.set_facecolor('white')

    ax = axes[0]
    levels = np.linspace(np.nanmin(Zp), np.nanmax(Zp), 15)
    cf = ax.contourf(XG, YG, Zp, levels=levels, cmap=cmap)
    cl = ax.contour(XG, YG, Zp, levels=levels[::2], colors='k', linewidths=0.5, alpha=0.5)
    ax.clabel(cl, inline=True, fontsize=6, fmt='%.2f')
    ax.scatter(coords_xy[:, 0], coords_xy[:, 1], c='navy', s=20, edgecolors='white',
               linewidths=0.6, zorder=5, label='Drill holes')
    plt.colorbar(cf, ax=ax, label=value_label)
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_title(f'Iso-{value_label.lower()} contour map (CZM, Jensen-corrected)', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right'); ax.set_aspect('equal')

    ax = axes[1]
    class_grid = np.where(active, np.where(Z_BT >= cutoff, Z_BT, np.nan), np.nan)
    cf2 = ax.contourf(XG, YG, class_grid, levels=15, cmap=cmap)
    below = active & (Z_BT < cutoff)
    ax.scatter(XG[below], YG[below], c='lightgray', s=8, marker='s', label=f'< {cutoff} cutoff')
    ax.scatter(coords_xy[:, 0], coords_xy[:, 1], c='navy', s=20, edgecolors='white', linewidths=0.6, zorder=5)
    plt.colorbar(cf2, ax=ax, label=value_label)
    n_res = int(np.sum(active & (Z_BT >= cutoff)))
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_title(f'Reserve classification (>= {cutoff}, n={n_res})', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right'); ax.set_aspect('equal')

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return out_path


def plot_isopach_isocal(XG, YG, THICK_BT, CAL_BT, active, coords_xy, cutoff, out_path, density=1.45):
    """4-panel figure for a dual-variable (thickness+calorific) deposit:
    isopach, iso-calorific, block-tonnage distribution, reserve classification."""
    TH = np.where(active, THICK_BT, np.nan)
    CAL = np.where(active, CAL_BT, np.nan)
    step = XG[0, 1] - XG[0, 0]

    fig, axes = plt.subplots(2, 2, figsize=(13, 11))
    fig.patch.set_facecolor('white')

    ax = axes[0, 0]
    levels = np.linspace(np.nanmin(TH), np.nanmax(TH), 15)
    cf = ax.contourf(XG, YG, TH, levels=levels, cmap='YlOrRd')
    cl = ax.contour(XG, YG, TH, levels=levels[::2], colors='k', linewidths=0.5, alpha=0.5)
    ax.clabel(cl, inline=True, fontsize=6, fmt='%.0f')
    ax.scatter(coords_xy[:, 0], coords_xy[:, 1], c='navy', s=8, alpha=0.6, zorder=5, label='Drill holes')
    plt.colorbar(cf, ax=ax, label='Thickness')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_title('Isopach map (CZM, Jensen-corrected)', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7, loc='lower left'); ax.set_aspect('equal')

    ax = axes[0, 1]
    levels2 = np.linspace(np.nanmin(CAL), np.nanmax(CAL), 15)
    cf2 = ax.contourf(XG, YG, CAL, levels=levels2, cmap='RdYlGn')
    cl2 = ax.contour(XG, YG, CAL, levels=levels2[::2], colors='k', linewidths=0.5, alpha=0.5)
    ax.clabel(cl2, inline=True, fontsize=6, fmt='%.0f')
    ax.scatter(coords_xy[:, 0], coords_xy[:, 1], c='navy', s=8, alpha=0.6, zorder=5)
    plt.colorbar(cf2, ax=ax, label='Calorific value')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_title('Iso-calorific map (CZM)', fontsize=10, fontweight='bold')
    ax.set_aspect('equal')

    ax = axes[1, 0]
    tonnage_block = TH * step * step * density / 1e6
    cf3 = ax.contourf(XG, YG, tonnage_block, levels=15, cmap='Blues')
    ax.scatter(coords_xy[:, 0], coords_xy[:, 1], c='navy', s=8, alpha=0.6, zorder=5)
    plt.colorbar(cf3, ax=ax, label='Tonnage per block (Mt)')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_title('Block tonnage distribution', fontsize=10, fontweight='bold')
    ax.set_aspect('equal')

    ax = axes[1, 1]
    class_grid = np.where(active, np.where(CAL >= cutoff, CAL, np.nan), np.nan)
    cf4 = ax.contourf(XG, YG, class_grid, levels=15, cmap='RdYlGn', vmin=np.nanmin(CAL), vmax=np.nanmax(CAL))
    below = active & (CAL < cutoff)
    ax.scatter(XG[below], YG[below], c='lightgray', s=25, marker='s', label=f'< {cutoff:.0f} cutoff')
    ax.scatter(coords_xy[:, 0], coords_xy[:, 1], c='navy', s=8, alpha=0.6, zorder=5)
    plt.colorbar(cf4, ax=ax, label='Calorific value')
    n_res = int(np.sum(active & (CAL >= cutoff)))
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_title(f'Reserve classification (>= {cutoff:.0f}, n={n_res})', fontsize=10, fontweight='bold')
    ax.legend(fontsize=7, loc='lower left'); ax.set_aspect('equal')

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return out_path


def plot_3d_block_bars(XG, YG, Z_BT, active, coords_xy, out_path, cmap='YlOrRd',
                        value_label='Value', title='3D block model (CZM)', step=None):
    """Pseudo-3D bar visualization (height proportional to value, flat base)
    -- appropriate when there is no true roof/floor surface (e.g. Kalgoorlie,
    a single composite grade per hole with no natural top/bottom geometry)."""
    if step is None:
        step = XG[0, 1] - XG[0, 0]
    fig = plt.figure(figsize=(10, 8))
    fig.patch.set_facecolor('white')
    ax = fig.add_subplot(111, projection='3d')
    vmin, vmax = np.nanmin(Z_BT[active]), np.nanmax(Z_BT[active])
    for i in range(XG.shape[0]):
        for j in range(XG.shape[1]):
            if active[i, j] and not np.isnan(Z_BT[i, j]):
                h = Z_BT[i, j]
                color = plt.colormaps[cmap]((Z_BT[i, j] - vmin) / (vmax - vmin + 1e-9))
                ax.bar3d(XG[i, j] - step / 2, YG[i, j] - step / 2, 0, step * 0.9, step * 0.9, h,
                         color=color, shade=True, edgecolor='none')
    ax.scatter(coords_xy[:, 0], coords_xy[:, 1], np.zeros(len(coords_xy)), c='navy', s=15, zorder=10)
    ax.set_xlabel('X', fontsize=8, labelpad=8)
    ax.set_ylabel('Y', fontsize=8, labelpad=8)
    ax.set_zlabel(value_label, fontsize=8, labelpad=6)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.view_init(elev=32, azim=-60)
    ax.tick_params(labelsize=7)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.1)
    cbar.set_label(value_label, fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return out_path


def plot_true_3d_block_model(XG, YG, roof, thickness, color_values, active, coords_xy, coords_roof,
                              out_path, color_label='Calorific value', cmap='RdYlGn', step=None):
    """TRUE 3D block model with real geological geometry: block roof = the
    (Delaunay-interpolated) top surface elevation, block floor = roof -
    thickness. Appropriate for a tabular seam deposit (Seyitömer) where a
    genuine roof/floor exists, unlike Kalgoorlie's single-composite-value
    gold assay data."""
    if step is None:
        step = XG[0, 1] - XG[0, 0]
    floor = roof - thickness
    valid = active & ~np.isnan(roof) & ~np.isnan(thickness)
    vmin, vmax = np.nanmin(color_values[valid]), np.nanmax(color_values[valid])
    fig = plt.figure(figsize=(11, 9))
    fig.patch.set_facecolor('white')
    ax = fig.add_subplot(111, projection='3d')
    for i in range(XG.shape[0]):
        for j in range(XG.shape[1]):
            if valid[i, j]:
                z0 = floor[i, j]; h = thickness[i, j]
                color = plt.colormaps[cmap]((color_values[i, j] - vmin) / (vmax - vmin + 1e-9))
                ax.bar3d(XG[i, j] - step / 2, YG[i, j] - step / 2, z0, step * 0.9, step * 0.9, h,
                         color=color, shade=True, edgecolor='none', alpha=0.95)
    ax.scatter(coords_xy[:, 0], coords_xy[:, 1], coords_roof, c='navy', s=8, alpha=0.5, zorder=10,
               label='Drill holes (roof)')
    ax.set_xlabel('X', fontsize=8, labelpad=8)
    ax.set_ylabel('Y', fontsize=8, labelpad=8)
    ax.set_zlabel('Elevation', fontsize=8, labelpad=6)
    ax.set_title('True 3D block model (roof=surveyed top, floor=roof-thickness)', fontsize=10, fontweight='bold')
    ax.view_init(elev=22, azim=-55)
    ax.tick_params(labelsize=7)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.1)
    cbar.set_label(color_label, fontsize=8)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return out_path


print("Cell loaded: build_block_grid, krige_grid, jensen_correct_log/sqrt, "
      "compute_reserve_single/dual, plot_isograde_map, plot_isopach_isocal, "
      "plot_3d_block_bars, plot_true_3d_block_model")


## 7. Driver — Kalgoorlie (fully reproducible)

Runs the complete pipeline: full benchmark table (old heuristic + 4 formulations
× 3 kernels), all figures, and the CZM-based resource estimate. Uses only the
public Kalgoorlie dataset already defined in Cell 1.

In [ ]:
"""
Full Kalgoorlie driver: fits the old heuristic + all 4 new formulations
across all 3 kernels, prints the complete benchmark table, generates all
figures, and computes the CZM-based resource estimate. Requires Cell 1
(KalgoorlieData, EmpiricalVariogram, etc.) to already be run.
"""

import os
os.makedirs("figures", exist_ok=True)

# ── Data & empirical variograms (Cell 1) ────────────────────────────────
data = KalgoorlieData()
ev = EmpiricalVariogram(data)
ev.compute_all()
ev.fit_all()

# ── Full benchmark: old heuristic (reference) + 4 new formulations × 3 kernels ──
print("=" * 90)
print("KALGOORLIE — FULL BENCHMARK (old heuristic reference + 4 formulations × 3 kernels)")
print("=" * 90)

kalgoorlie_results = []
for kernel in ['spherical', 'exponential', 'gaussian']:
    m = OldVSKHeuristic(ev, kernel=kernel)
    Cm = PSDAnalysis.build_matrix(data.coords, m.cov_func, m.cov_diagonal())
    p = PSDAnalysis.check(Cm, f"Old-VSK [{kernel}]")
    ok = OrdinaryKriging(m)
    r = ok.loocv(data.coords, data.au_log, m.cov_func, f"Old-VSK [{kernel}]")
    print(f"  {'Old-VSK [' + kernel + ']':<28} n_par={m.n_params:2d}  "
          f"lambda_min={p['lmin']:12.5g}  kappa={p['cond']:9.1f}  "
          f"PSD={'Y' if p['psd'] else 'N'}  R2={r['R2']:8.4f}  RMSE={r['RMSE']:9.4f}")
    kalgoorlie_results.append({"method": f"Old-VSK [{kernel}]", "n_params": m.n_params,
                               "AICc": None, "lambda_min": p["lmin"], "kappa": p["cond"],
                               "PSD": p["psd"], "R2": r["R2"], "RMSE": r["RMSE"]})

kalgoorlie_results += run_benchmark(ev, data.coords, data.au_log, n_restarts=30, seed=42)

# best model per formulation (by AICc, excluding the reference row) for figures/resource
best_by_method = {}
for row in kalgoorlie_results:
    base = row["method"].split(" [")[0]
    if base == "Old-VSK":
        continue
    if row["AICc"] is not None and (base not in best_by_method or row["AICc"] < best_by_method[base][1]):
        best_by_method[base] = (row["method"], row["AICc"])
print("\nBest kernel per formulation (by AICc):")
for base, (full_label, aicc) in best_by_method.items():
    print(f"  {base:<20} -> {full_label}  (AICc={aicc:.2f})")

# ── Refit the best-per-method models for figures & resource estimation ──
best_kernel = {base: full_label.split("[")[1].rstrip("]") for base, (full_label, _) in best_by_method.items()}
te = TrueEllipticalAnisotropic(ev, kernel=best_kernel.get('3D-VSK-Elliptical', 'spherical')); te.fit(n_restarts=30, seed=42)
lmr = NestedLMR(ev, kernel=best_kernel.get('Nested-LMR', 'spherical')); lmr.fit(n_restarts=30, seed=42)
czm = ContinuousZonalMixture(ev, K=1, kernel=best_kernel.get('CZM', 'spherical')); czm.fit(n_restarts=30, seed=42, bound=5.0)
ksm = KernelSumMixture(ev, J=4, kernel=best_kernel.get('KernelSum', 'spherical')); ksm.fit(n_restarts=20, seed=42)

def nearest_dir(az, dirs):
    base = az % 180
    diffs = [min(abs(base - d), 180 - abs(base - d)) for d in dirs]
    return dirs[np.argmin(diffs)]

def cov_discrete(xi, xj):
    if np.allclose(xi, xj):
        return ev.fitted[0]['C0'] + ev.fitted[0]['C']
    h = np.linalg.norm(xi[:2] - xj[:2])
    az = np.degrees(np.arctan2(xj[1] - xi[1], xj[0] - xi[0])) % 180
    d = nearest_dir(az, ev.DIRECTIONS)
    p = ev.fitted[d]
    return VariogramModel.spherical_cov(h, p['C0'], p['C'], p['a'])

class DiscreteModel:
    def cov_func(self, xi, xj): return cov_discrete(xi, xj)
    def cov_diagonal(self, params=None): return ev.fitted[0]['C0'] + ev.fitted[0]['C']
    def gamma_model(self, h, theta_deg, params=None):
        d = nearest_dir(theta_deg, ev.DIRECTIONS)
        p = ev.fitted[d]
        return VariogramModel.spherical_gamma(np.atleast_1d(h), p['C0'], p['C'], p['a'])
discrete_model = DiscreteModel()

# ── Figures ──────────────────────────────────────────────────────────────
print("\nGenerating figures...")
bench_order = [('3D-VSK-Elliptical [' + best_kernel.get('3D-VSK-Elliptical','spherical') + ']',
                '3D-VSK-\nElliptical', '#e67e22'),
               ('Nested-LMR [' + best_kernel.get('Nested-LMR','spherical') + ']', 'Nested-\nLMR', '#8e44ad'),
               ('CZM [' + best_kernel.get('CZM','spherical') + ']', 'CZM', '#2980b9'),
               ('KernelSum [' + best_kernel.get('KernelSum','spherical') + ']', 'KernelSum', '#16a085')]
plot_benchmark_bars(kalgoorlie_results, bench_order, "figures/kalgoorlie_fig_benchmark.png", rmse_label='RMSE (log space)')

plot_surface_3d([('3D-VSK-Elliptical', te), ('CZM', czm), ('Nested-LMR', lmr), ('KernelSum', ksm)],
                h_max=68, out_path="figures/kalgoorlie_fig_surface3d.png")

plot_contour_cross_section([('3D-VSK-Elliptical', te), ('CZM', czm), ('Nested-LMR', lmr), ('KernelSum', ksm)],
                            h_max=68, out_path="figures/kalgoorlie_fig_contour_cut.png")

plot_eigenvalue_spectrum([('Discrete', discrete_model.cov_func, discrete_model.cov_diagonal()),
                          ('3D-VSK-Elliptical', te.cov_func, te.cov_diagonal()),
                          ('CZM', czm.cov_func, czm.cov_diagonal()),
                          ('Nested-LMR', lmr.cov_func, lmr.cov_diagonal())],
                         data.coords, "figures/kalgoorlie_fig_eigenspectrum.png")

plot_loocv_scatter([('Discrete', discrete_model.cov_func, discrete_model.cov_diagonal()),
                    ('3D-VSK-Elliptical', te.cov_func, te.cov_diagonal()),
                    ('CZM', czm.cov_func, czm.cov_diagonal()),
                    ('Nested-LMR', lmr.cov_func, lmr.cov_diagonal())],
                   data.coords, data.au_log, "figures/kalgoorlie_fig_loocv_scatter.png",
                   obs_label='Observed ln(Au)')
print("  Figures saved to figures/kalgoorlie_fig_*.png")

# ── Resource estimation (CZM) ────────────────────────────────────────────
print("\nResource estimation (CZM)...")
XG, YG, active, tree = build_block_grid(data.coords[:, :2], step=2.0, search_radius=45.0)
z_level = data.coords[:, 2].mean()
Z_HAT, SIG2 = krige_grid(czm, XG, YG, z_level, active, data.coords, data.au_log, tree, search_radius=45.0)
Z_BT = jensen_correct_log(Z_HAT, SIG2)
kalgoorlie_reserve = compute_reserve_single(Z_BT, active, step=2.0, density=2.7, cutoff=0.5)
print("  Reserve:", kalgoorlie_reserve)

plot_isograde_map(XG, YG, Z_BT, active, data.coords[:, :2], cutoff=0.5,
                   out_path="figures/kalgoorlie_fig_isograde.png", value_label='Au grade (g/t)')
plot_3d_block_bars(XG, YG, Z_BT, active, data.coords[:, :2], out_path="figures/kalgoorlie_fig_3dblock.png",
                    value_label='Au grade (g/t)', title='3D block model — Kalgoorlie Au (CZM)', step=2.0)
print("  Resource figures saved.")
print("\nKalgoorlie driver complete.")


## 8. Driver — Seyitömer (requires confidential real dataset)

**Requires** `SLI_ASLANLI_DATABASE.xlsx` (columns: X, Y, thickness,
calorific_value, coal_top) — available from the corresponding author upon
reasonable request; not distributed publicly for confidentiality reasons.
Update `DATA_PATH` below if the file is stored elsewhere.

Runs the complete pipeline for BOTH variables (sqrt-thickness and calorific
value): full benchmark table, all figures, and a joint CZM-based resource
estimate (dual-variable, true 3D block geometry using the interpolated
`coal_top` roof surface).

In [ ]:
"""
Full Seyitömer driver: runs the complete pipeline (benchmark, figures,
resource estimation) for BOTH variables (sqrt-thickness and calorific
value) using the real drill-hole dataset.

REQUIRES: SLI_ASLANLI_DATABASE.xlsx (X, Y, thickness, calorific_value,
coal_top columns), available from the corresponding author upon reasonable
request (see the manuscript's Data Availability Statement) -- not included
in the public repository for confidentiality reasons.
"""

import os
import numpy as np
import pandas as pd
from scipy.interpolate import LinearNDInterpolator

os.makedirs("figures", exist_ok=True)

DATA_PATH = "SLI_ASLANLI_DATABASE.xlsx"   # update path as needed
df = pd.read_excel(DATA_PATH)
coords_xy = df[['X', 'Y']].values
coords3 = df[['X', 'Y']].assign(z=df['coal_top']).values


def run_seyitomer_variable(values, var_name, obs_label, n_restarts=20, k_pp=20):
    print("=" * 90)
    print(f"SEYITOMER — {var_name}: FULL BENCHMARK (old heuristic reference + 4 formulations × 3 kernels)")
    print("=" * 90)

    data_v = GenericFieldData(coords3, values)
    ev_v = GenericEmpiricalVariogram(data_v)
    ev_v.compute_all()
    ev_v.fit_all()

    var_results = []
    for kernel in ['spherical', 'exponential', 'gaussian']:
        m = OldVSKHeuristic(ev_v, kernel=kernel)
        Cm = PSDAnalysis.build_matrix(coords3, m.cov_func, m.cov_diagonal())
        p = PSDAnalysis.check(Cm, f"Old-VSK [{kernel}]")
        loo = FastNeighborLOOCV(coords3, values, m.cov_func, m.cov_diagonal(), k_pp=k_pp)
        r = loo.run(f"Old-VSK [{kernel}]")
        print(f"  {'Old-VSK [' + kernel + ']':<28} n_par={m.n_params:2d}  "
              f"lambda_min={p['lmin']:14.5g}  kappa={p['cond']:9.1f}  "
              f"PSD={'Y' if p['psd'] else 'N'}  R2={r['R2']:8.4f}  RMSE={r['RMSE']:9.4f}")
        var_results.append({"method": f"Old-VSK [{kernel}]", "n_params": m.n_params, "AICc": None,
                             "lambda_min": p["lmin"], "kappa": p["cond"], "PSD": p["psd"],
                             "R2": r["R2"], "RMSE": r["RMSE"]})

    var_results += run_benchmark(ev_v, coords3, values, n_restarts=n_restarts, k_pp=k_pp, seed=42)

    best_by_method = {}
    for row in var_results:
        base = row["method"].split(" [")[0]
        if base == "Old-VSK":
            continue
        if row["AICc"] is not None and (base not in best_by_method or row["AICc"] < best_by_method[base][1]):
            best_by_method[base] = (row["method"], row["AICc"])
    print("\nBest kernel per formulation (by AICc):")
    for base, (full_label, aicc) in best_by_method.items():
        print(f"  {base:<20} -> {full_label}  (AICc={aicc:.2f})")
    best_kernel = {base: full_label.split("[")[1].rstrip("]") for base, (full_label, _) in best_by_method.items()}

    te = TrueEllipticalAnisotropic(ev_v, kernel=best_kernel.get('3D-VSK-Elliptical', 'spherical'))
    te.fit(n_restarts=n_restarts, seed=42)
    lmr = NestedLMR(ev_v, kernel=best_kernel.get('Nested-LMR', 'spherical'))
    lmr.fit(n_restarts=n_restarts, seed=42)
    czm = ContinuousZonalMixture(ev_v, K=1, kernel=best_kernel.get('CZM', 'spherical'))
    czm.fit(n_restarts=n_restarts, seed=42, bound=5.0)
    ksm = KernelSumMixture(ev_v, J=4, kernel=best_kernel.get('KernelSum', 'spherical'))
    ksm.fit(n_restarts=max(6, n_restarts // 2), seed=42)

    print("\nGenerating figures...")
    tag = var_name.lower().replace(" ", "_")
    bench_order = [(f'3D-VSK-Elliptical [{best_kernel.get("3D-VSK-Elliptical","spherical")}]',
                     '3D-VSK-\nElliptical', '#e67e22'),
                    (f'Nested-LMR [{best_kernel.get("Nested-LMR","spherical")}]', 'Nested-\nLMR', '#8e44ad'),
                    (f'CZM [{best_kernel.get("CZM","spherical")}]', 'CZM', '#2980b9'),
                    (f'KernelSum [{best_kernel.get("KernelSum","spherical")}]', 'KernelSum', '#16a085')]
    plot_benchmark_bars(var_results, bench_order, f"figures/seyitomer_{tag}_fig_benchmark.png")

    h_max = max(ev_v.fitted[d]['a'] for d in ev_v.DIRECTIONS) * 1.05
    plot_surface_3d([('3D-VSK-Elliptical', te), ('CZM', czm), ('Nested-LMR', lmr), ('KernelSum', ksm)],
                     h_max=h_max, out_path=f"figures/seyitomer_{tag}_fig_surface3d.png")
    plot_contour_cross_section([('3D-VSK-Elliptical', te), ('CZM', czm), ('Nested-LMR', lmr), ('KernelSum', ksm)],
                                h_max=h_max, out_path=f"figures/seyitomer_{tag}_fig_contour_cut.png")
    plot_eigenvalue_spectrum([('3D-VSK-Elliptical', te.cov_func, te.cov_diagonal()),
                              ('CZM', czm.cov_func, czm.cov_diagonal()),
                              ('Nested-LMR', lmr.cov_func, lmr.cov_diagonal()),
                              ('KernelSum', ksm.cov_func, ksm.cov_diagonal())],
                             coords3, f"figures/seyitomer_{tag}_fig_eigenspectrum.png")
    plot_loocv_scatter([('3D-VSK-Elliptical', te.cov_func, te.cov_diagonal()),
                        ('CZM', czm.cov_func, czm.cov_diagonal()),
                        ('Nested-LMR', lmr.cov_func, lmr.cov_diagonal()),
                        ('KernelSum', ksm.cov_func, ksm.cov_diagonal())],
                       coords3, values, f"figures/seyitomer_{tag}_fig_loocv_scatter.png",
                       obs_label=obs_label, k_pp=k_pp)
    print(f"  Figures saved to figures/seyitomer_{tag}_fig_*.png")

    return {"results": var_results, "models": {"te": te, "lmr": lmr, "czm": czm, "ksm": ksm},
            "best_kernel": best_kernel}


# ── Run both variables ───────────────────────────────────────────────────
thickness_out = run_seyitomer_variable(np.sqrt(df['thickness'].values), "sqrtThickness",
                                        obs_label='Observed sqrt(Thickness)')
calorific_out = run_seyitomer_variable(df['calorific_value'].values, "Calorific",
                                        obs_label='Observed calorific value (kcal/kg)')

# ── Resource estimation (CZM, both variables jointly) ────────────────────
print("\n" + "=" * 90)
print("SEYITOMER — RESOURCE ESTIMATION (CZM, thickness + calorific jointly)")
print("=" * 90)

czm_t = thickness_out["models"]["czm"]
czm_c = calorific_out["models"]["czm"]

XG, YG, active, tree = build_block_grid(coords_xy, step=200.0, search_radius=600.0)
z_level = coords3[:, 2].mean()
TH_HAT, TH_SIG2 = krige_grid(czm_t, XG, YG, z_level, active, coords3, np.sqrt(df['thickness'].values),
                              tree, search_radius=600.0)
CAL_HAT, CAL_SIG2 = krige_grid(czm_c, XG, YG, z_level, active, coords3, df['calorific_value'].values,
                                tree, search_radius=600.0)
THICK_BT = jensen_correct_sqrt(TH_HAT, TH_SIG2)
CAL_BT = CAL_HAT

seyitomer_reserve = compute_reserve_dual(THICK_BT, CAL_BT, active, step=200.0, density=1.45, cutoff=2200.0)
print("Reserve:", seyitomer_reserve)

plot_isopach_isocal(XG, YG, THICK_BT, CAL_BT, active, coords_xy, cutoff=2200.0,
                     out_path="figures/seyitomer_fig_isopach_isocal.png")

interp_top = LinearNDInterpolator(coords_xy, df['coal_top'].values)
ROOF = interp_top(XG, YG)
plot_true_3d_block_model(XG, YG, ROOF, THICK_BT, CAL_BT, active, coords_xy, df['coal_top'].values,
                          out_path="figures/seyitomer_fig_true3dblock.png", color_label='Calorific value')

print("Resource figures saved.")
print("\nSeyitömer driver complete.")


## 9. Extended Synthetic PSD Counterexample (Fig. 1)

Demonstrates, on the manuscript's own 3-point synthetic configuration
(Section 2.5.2 / Supplementary S1), that all four new formulations remain
PSD across the full tested range of the range-ratio r = a1/a2, unlike
Classical Discrete which loses PSD beyond r* ≈ 5.71.

In [ ]:
"""
Extended synthetic PSD counterexample (manuscript Fig. 1 / Supplementary S1),
now showing that all four new formulations remain PSD across the full range
of r = a1/a2 tested, unlike Classical Discrete which loses PSD beyond
r* ≈ 5.71. Uses the exact 3-point configuration from the manuscript's S1
derivation (P1=(0,0), P2=(0,0.8*a2), P3=(0.14*a1,0), a2=10 m fixed).
"""

import numpy as np
import matplotlib.pyplot as plt

a2 = 10.0


def _points(r):
    a1 = r * a2
    return [np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.8 * a2, 0.0]), np.array([0.14 * a1, 0.0, 0.0])]


def _lmin_discrete(r):
    a1 = r * a2
    def nearest_dir(az):
        base = az % 180
        return 0 if min(abs(base - 0), 180 - abs(base - 0)) <= min(abs(base - 90), 180 - abs(base - 90)) else 90
    def cov(xi, xj):
        if np.allclose(xi, xj):
            return 1.0
        dx, dy = xj[0] - xi[0], xj[1] - xi[1]
        h = np.hypot(dx, dy)
        az = np.degrees(np.arctan2(dy, dx)) % 180
        a = a1 if nearest_dir(az) == 0 else a2
        return VariogramModel.spherical_cov(h, 0.0, 1.0, a)
    pts = _points(r)
    K = np.array([[cov(pts[i], pts[j]) for j in range(3)] for i in range(3)])
    return np.linalg.eigvalsh(K).min()


def _lmin_generic(r, cov_func, diag_val=1.0):
    pts = _points(r)
    K = np.array([[diag_val if i == j else cov_func(pts[i], pts[j]) for j in range(3)] for i in range(3)])
    return np.linalg.eigvalsh(K).min()


def make_extended_fig1(out_path="figures/fig1_psd_counterexample_extended.png", kernel='spherical'):
    r_values = np.concatenate([np.linspace(1, 10, 25), np.linspace(10, 25, 15)])
    curves = {'Discrete': [], '3D-VSK-Elliptical': [], 'CZM': [], 'Nested-LMR': [], 'KernelSum': []}

    for r in r_values:
        a1 = r * a2
        curves['Discrete'].append(_lmin_discrete(r))

        te = TrueEllipticalAnisotropic.__new__(TrueEllipticalAnisotropic)
        te.kernel = kernel
        te.params = np.array([0.0, 1.0, a1, a2, 0.0])
        curves['3D-VSK-Elliptical'].append(_lmin_generic(r, te.cov_func, te.cov_diagonal()))

        czm = ContinuousZonalMixture.__new__(ContinuousZonalMixture)
        czm.K = 1; czm.kernel = kernel
        czm.params = np.array([0.0, 0.25, a2, a1, 0.75, 1.5, 0.0])
        curves['CZM'].append(_lmin_generic(r, czm.cov_func, czm.cov_diagonal()))

        lmr = NestedLMR.__new__(NestedLMR)
        lmr.kernel = kernel; lmr._a_max_z = 500 * a2
        lmr.params = np.array([0.0, np.log(0.7), a1, a2, 0.0, np.log(0.3), a2, 90.0])
        curves['Nested-LMR'].append(_lmin_generic(r, lmr.cov_func, lmr.cov_diagonal()))

        ksm = KernelSumMixture.__new__(KernelSumMixture)
        ksm.J = 4; ksm.kernel = kernel
        ksm.mu = np.array([j * np.pi / 4 for j in range(4)])
        ksm.params = np.array([0.0, 0.2, a2, a1, 3.0, np.log(0.7), np.log(0.02), np.log(0.02), np.log(0.02)])
        curves['KernelSum'].append(_lmin_generic(r, ksm.cov_func, ksm.cov_diagonal()))

    fig, ax = plt.subplots(1, 1, figsize=(8, 5.5))
    fig.patch.set_facecolor('white')
    styles = {'Discrete': ('#c0392b', '--', 2.2), '3D-VSK-Elliptical': ('#e67e22', '-', 2.0),
              'CZM': ('#2980b9', '-', 2.0), 'Nested-LMR': ('#8e44ad', '-', 2.0),
              'KernelSum': ('#16a085', '-', 2.0)}
    for name, (col, ls, lw) in styles.items():
        ax.plot(r_values, curves[name], ls, color=col, lw=lw, label=name)
    ax.axhline(0, color='black', lw=1, alpha=0.6)
    ax.axvline(5.71, color='gray', ls=':', lw=1.3)
    ax.text(5.71, ax.get_ylim()[1] * 0.0, ' r*=5.71', fontsize=8, color='gray', va='bottom')
    ax.set_xlabel('Range ratio r = a1/a2', fontsize=10)
    ax.set_ylabel('lambda_min', fontsize=10)
    ax.set_title('Synthetic PSD counterexample: lambda_min vs. range ratio (all methods)',
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=9, loc='center right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved -> {out_path}")
    for name in curves:
        print(f"  {name:<20} min(lambda_min) over r in [1,25] = {min(curves[name]):.5f}")
    return curves


if __name__ == "__main__":
    make_extended_fig1()


make_extended_fig1()
